In [ ]:
# 0. 실행 설정: 날짜와 ROI는 여기만 수정
TARGET_DATE = globals().get("TARGET_DATE", "2007-12-15")
ROI_LABEL = globals().get("ROI_LABEL", "manual_roi")
ROI_CENTER_LAT = globals().get("ROI_CENTER_LAT", 37.3860959)
ROI_CENTER_LON = globals().get("ROI_CENTER_LON", 126.6389202)

# 배치 실행에서만 사용. 일반 단일 실행이면 None 그대로 둔다.
RUNS_ROOT_DIR = globals().get("RUNS_ROOT_DIR", None)
PRESELECTED_LANDSAT_SCENE = globals().get("PRESELECTED_LANDSAT_SCENE", None)


## 1. Condition 생성


In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import json, math, os, re, time
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageDraw
from pyproj import Transformer
import ee

try:
    import rasterio
    from rasterio.enums import Resampling
except Exception:
    rasterio = None
    Resampling = None


# ROI/날짜는 0번 설정 셀에서 바꾼다. 0번 셀을 건너뛰었을 때만 아래 기본값을 쓴다.
TARGET_DATE = globals().get("TARGET_DATE", "2007-04-15")
ROI_LABEL = globals().get("ROI_LABEL", "manual_roi")
ROI_CENTER_LAT = globals().get("ROI_CENTER_LAT", 37.3860959)
ROI_CENTER_LON = globals().get("ROI_CENTER_LON", 126.6389202)
PRESELECTED_LANDSAT_SCENE = globals().get("PRESELECTED_LANDSAT_SCENE", None)

# 요청 날짜에 Landsat 장면이 없으면 요청일 이전/당일 중 가장 최근 장면을 쓴다.
LANDSAT_LOOKBACK_DAYS = 3650
MAX_SCENE_CLOUD_COVER = 20
LANDSAT_COLLECTIONS = [
    "LANDSAT/LT05/C02/T1_L2",
    "LANDSAT/LC08/C02/T1_L2",
    "LANDSAT/LC09/C02/T1_L2",
]

# EGIS unknown 처리는 fill/save 셀 실행 시 입력받는다. 1=바다, 2=도심
OVERWRITE = bool(globals().get("OVERWRITE", False))

# 외부 API 설정: 환경변수가 없으면 고정 키를 바로 사용한다.
EE_PROJECT = os.environ.get("EE_PROJECT", "neat-acre-496003-i5")
DEFAULT_KMA_AUTH_KEY = "34j6vBWrSGiI-rwVq0hoRg"
KMA_AUTH_KEY = os.environ.get("KMA_AUTH_KEY") or os.environ.get("KMA_APIHUB_AUTH_KEY") or DEFAULT_KMA_AUTH_KEY
REQUEST_TIMEOUT = 180


PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != "26.05_MLproj" and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent
if PROJECT_DIR.name != "26.05_MLproj":
    PROJECT_DIR = Path("/mnt/c/Users/shinh/eigenspace/Workspace/26.05_MLproj")

RECONST_DIR = PROJECT_DIR / "reconst"
RUNS_ROOT_BASE = Path(globals().get("RUNS_ROOT_DIR") or (RECONST_DIR / "runs"))


def run_token(value):
    return str(value).replace("/", "_").replace(chr(92), "_").replace(":", "_").replace(" ", "_").replace(".", "p")


REQUESTED_DATE_LABEL = pd.Timestamp(TARGET_DATE).strftime("%Y-%m-%d")
RUN_LABEL = None
RUN_ROOT_DIR = None
CONDITION_DIR = None
RECONSTRUCTION_DIR = None
OUT_DIR = None
VIS_DIR = None
SCRATCH_DIR = None
REVIEW_DIR = None
DIRECT_RAW_DIR = None
REVIEW_CSV = None
DECISION_CSV = None

GRID_N = 256
PIXEL_SIZE_M = 60
CRS_BASE = "EPSG:32653"
NODATA = -9999.0
HOUR_KST = "1100"
LST_VALID_MIN_K = 150.0
LST_VALID_MAX_K = 380.0
ALBEDO_FALLBACK_VALUE = 0.18
ALBEDO_FILL_RADII_PX = (1, 2, 4, 8, 16, 32, 64, 128)
TERRAIN_BANDS = ["elevation", "slope", "aspect_sin", "aspect_cos", "hillshade_landsat"]
ALBEDO_BANDS = ["albedo"]
AWS_BANDS = ["avg_temp", "wind_u", "wind_v", "rain_1h", "humidity"]
META_CHANNELS = ["day_sin", "day_cos", "sun_azimuth_norm", "sun_elevation_norm", "lat_norm", "lon_norm"]
CONDITION_CHANNELS = TERRAIN_BANDS + ALBEDO_BANDS + AWS_BANDS + [
    "egis_cat_110_urban_residential", "egis_cat_120_urban_industrial", "egis_cat_130_urban_commercial",
    "egis_cat_140_urban_cultural_public", "egis_cat_150_urban_transportation", "egis_cat_160_urban_mixed",
    "egis_cat_200_agriculture", "egis_cat_300_forest", "egis_cat_400_grassland", "egis_cat_500_wetland",
    "egis_cat_600_barren", "egis_cat_710_water_inland", "egis_cat_720_water_marine",
]

META_NORM_SOURCE = "reconst_fixed_training_meta_scale"
LAT_META_MEAN = 36.22751454452666
LAT_META_STD = 0.974472059049708
LON_META_MEAN = 127.70786578208728
LON_META_STD = 0.8830399708103981


print("requested target_date:", REQUESTED_DATE_LABEL)
print("data source: direct EE + KMA + EGIS WMS")
print("meta norm source:", META_NORM_SOURCE)
print("lat/lon meta norm:", {"lat_mean": LAT_META_MEAN, "lat_std": LAT_META_STD, "lon_mean": LON_META_MEAN, "lon_std": LON_META_STD})


def make_patch_grid(center_lat, center_lon, grid_n=GRID_N, pixel_size_m=PIXEL_SIZE_M, crs_base=CRS_BASE):
    to_xy = Transformer.from_crs("EPSG:4326", crs_base, always_xy=True)
    cx, cy = to_xy.transform(float(center_lon), float(center_lat))
    half = grid_n * pixel_size_m / 2
    left, right = cx - half, cx + half
    bottom, top = cy - half, cy + half
    xs = left + (np.arange(grid_n) + 0.5) * pixel_size_m
    ys = top - (np.arange(grid_n) + 0.5) * pixel_size_m
    xx, yy = np.meshgrid(xs, ys)
    return {"n": grid_n, "pixel_size": pixel_size_m, "crs": crs_base, "left": left, "right": right, "bottom": bottom, "top": top, "xx": xx.astype(np.float32), "yy": yy.astype(np.float32)}

def ee_region_for_grid(grid):
    return ee.Geometry.Rectangle([grid["left"], grid["bottom"], grid["right"], grid["top"]], proj=grid["crs"], geodesic=False)

def validate_kma_auth_key(key):
    key = str(key).strip()
    if len(key) < 10 or "°" in key:
        raise ValueError("KMA APIHub authKey가 비어 있거나 잘못되었습니다.")
    return key

def initialize_external_clients():
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)
    global KMA_AUTH_KEY
    if not KMA_AUTH_KEY:
        raise ValueError("KMA APIHub authKey가 없습니다. DEFAULT_KMA_AUTH_KEY 또는 환경변수를 확인하세요.")
    KMA_AUTH_KEY = validate_kma_auth_key(KMA_AUTH_KEY)
    print("EE initialized; KMA key accepted")

initialize_external_clients()
grid = make_patch_grid(ROI_CENTER_LAT, ROI_CENTER_LON)
roi = ee_region_for_grid(grid)


def merged_landsat_collection(roi, start_date, end_date):
    parts = []
    for col_id in LANDSAT_COLLECTIONS:
        parts.append(
            ee.ImageCollection(col_id)
            .filterBounds(roi)
            .filterDate(str(start_date), str(end_date))
            .filter(ee.Filter.lt("CLOUD_COVER", float(MAX_SCENE_CLOUD_COVER)))
        )
    col = parts[0]
    for part in parts[1:]:
        col = col.merge(part)
    return col

def ee_scene_row(feature):
    props = feature["properties"]
    ts = pd.to_datetime(props["system:time_start"], unit="ms", utc=True).tz_convert("Asia/Seoul")
    return {
        "date": ts.strftime("%Y-%m-%d"),
        "timestamp_kst": ts.strftime("%Y-%m-%d %H:%M:%S%z"),
        "scene_id": feature["id"],
        "collection": "/".join(feature["id"].split("/")[:-1]),
        "landsat_product_id": props.get("LANDSAT_PRODUCT_ID") or feature["id"].split("/")[-1],
        "cloud_cover_scene": props.get("CLOUD_COVER"),
        "sun_azimuth": props.get("SUN_AZIMUTH"),
        "sun_elevation": props.get("SUN_ELEVATION"),
    }

def select_landsat_scene():
    requested = pd.Timestamp(TARGET_DATE)
    exact_start = requested.strftime("%Y-%m-%d")
    exact_end = (requested + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    recent_start = (requested - pd.Timedelta(days=int(LANDSAT_LOOKBACK_DAYS))).strftime("%Y-%m-%d")
    recent_end = exact_end
    exact_col = merged_landsat_collection(roi, exact_start, exact_end)
    exact_count = int(exact_col.size().getInfo())
    recent_col = merged_landsat_collection(roi, recent_start, recent_end).sort("system:time_start", False)
    recent_count = int(recent_col.size().getInfo())
    if recent_count <= 0:
        raise RuntimeError(f"{TARGET_DATE} 이전 {LANDSAT_LOOKBACK_DAYS}일 내 ROI와 겹치는 Landsat 장면이 없습니다.")
    recent_info = recent_col.limit(50).getInfo()
    recent_rows = [ee_scene_row(feature) for feature in recent_info["features"]]
    selected = recent_rows[0]
    selected_date_count = sum(1 for row in recent_rows if row["date"] == selected["date"])
    exact_rows = []
    if exact_count:
        exact_rows = [ee_scene_row(feature) for feature in exact_col.limit(20).getInfo()["features"]]
    return {
        "requested_date": requested.strftime("%Y-%m-%d"),
        "landsat_exact_available": bool(exact_count > 0),
        "landsat_exact_count": int(exact_count),
        "selected_landsat_date": selected["date"],
        "selected_landsat_date_count": int(selected_date_count),
        "selected_landsat_scene_id": selected["scene_id"],
        "selected_landsat_product_id": selected["landsat_product_id"],
        "selected_landsat_collection": selected["collection"],
        "selected_landsat_cloud_cover_scene": selected["cloud_cover_scene"],
        "selected_landsat_sun_azimuth": selected["sun_azimuth"],
        "selected_landsat_sun_elevation": selected["sun_elevation"],
        "exact_landsat_scenes": exact_rows,
        "recent_landsat_scenes_checked": recent_rows,
    }

def landsat_info_from_scene_row(scene_row, requested_date=None):
    selected_date = str(scene_row["date"])
    requested = pd.Timestamp(requested_date or selected_date).strftime("%Y-%m-%d")
    return {
        "requested_date": requested,
        "landsat_exact_available": requested == selected_date,
        "landsat_exact_count": int(1 if requested == selected_date else 0),
        "selected_landsat_date": selected_date,
        "selected_landsat_date_count": int(scene_row.get("selected_landsat_date_count", 1)),
        "selected_landsat_scene_id": str(scene_row["scene_id"]),
        "selected_landsat_product_id": str(scene_row["landsat_product_id"]),
        "selected_landsat_collection": str(scene_row["collection"]),
        "selected_landsat_cloud_cover_scene": scene_row.get("cloud_cover_scene"),
        "selected_landsat_sun_azimuth": scene_row.get("sun_azimuth"),
        "selected_landsat_sun_elevation": scene_row.get("sun_elevation"),
        "exact_landsat_scenes": [dict(scene_row)] if requested == selected_date else [],
        "recent_landsat_scenes_checked": [dict(scene_row)],
    }

landsat_info = landsat_info_from_scene_row(PRESELECTED_LANDSAT_SCENE, TARGET_DATE) if PRESELECTED_LANDSAT_SCENE else select_landsat_scene()
selected_date = pd.Timestamp(landsat_info["selected_landsat_date"])
analysis_year = int(selected_date.year)
row = {
    "date": landsat_info["selected_landsat_date"],
    "year": analysis_year,
    "latitude": float(ROI_CENTER_LAT),
    "longitude": float(ROI_CENTER_LON),
    "scene_id": landsat_info["selected_landsat_scene_id"],
    "collection": landsat_info["selected_landsat_collection"],
    "landsat_product_id": landsat_info["selected_landsat_product_id"],
    "sun_azimuth": float(landsat_info["selected_landsat_sun_azimuth"]),
    "sun_elevation": float(landsat_info["selected_landsat_sun_elevation"]),
}
print("requested_date:", landsat_info["requested_date"])
print("landsat_exact_available:", landsat_info["landsat_exact_available"], "count:", landsat_info["landsat_exact_count"])
print("selected_landsat_date:", landsat_info["selected_landsat_date"])
print("selected_landsat_product_id:", landsat_info["selected_landsat_product_id"])
print("selected_landsat_scene_id:", landsat_info["selected_landsat_scene_id"])

RUN_DATE_LABEL = landsat_info["selected_landsat_date"]
RUN_LABEL = f"{run_token(RUN_DATE_LABEL)}_{run_token(ROI_LABEL)}_lat{run_token(round(float(ROI_CENTER_LAT), 5))}_lon{run_token(round(float(ROI_CENTER_LON), 5))}"
RUN_ROOT_DIR = RUNS_ROOT_BASE / RUN_LABEL
CONDITION_DIR = RUN_ROOT_DIR / "outputs" / "condition"
RECONSTRUCTION_DIR = RUN_ROOT_DIR / "outputs" / "reconstruction"
OUT_DIR = CONDITION_DIR
VIS_DIR = RUN_ROOT_DIR / "outputs" / "visualization"
SCRATCH_DIR = RUN_ROOT_DIR / "_scratch"
REVIEW_DIR = SCRATCH_DIR / "unknown_review"
DIRECT_RAW_DIR = SCRATCH_DIR / "direct_raw"
REVIEW_CSV = REVIEW_DIR / "egis_unknown_review.csv"
DECISION_CSV = REVIEW_DIR / "egis_unknown_decisions.csv"
for p in [CONDITION_DIR, RECONSTRUCTION_DIR, VIS_DIR, REVIEW_DIR, DIRECT_RAW_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print("run_label:", RUN_LABEL)
print("run root:", RUN_ROOT_DIR)
print("condition output:", CONDITION_DIR)
print("visualization output:", VIS_DIR)
print("scratch:", SCRATCH_DIR)


def ee_stack(image, bands, grid):
    if rasterio is None:
        raise ImportError("rasterio가 필요합니다. 현재 환경에 설치 후 이 노트북을 실행하세요.")
    image = image.select(bands).toFloat().unmask(NODATA)
    transform = [grid["pixel_size"], 0, grid["left"], 0, -grid["pixel_size"], grid["top"]]
    params = {"name": "patch", "bands": bands, "crs": grid["crs"], "crs_transform": transform, "dimensions": [grid["n"], grid["n"]], "filePerBand": False, "format": "GEO_TIFF"}
    url = image.getDownloadURL(params)
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    data = r.content
    if data.startswith(b"PK"):
        with ZipFile(BytesIO(data)) as zf:
            names = [n for n in zf.namelist() if n.lower().endswith((".tif", ".tiff"))]
            if not names:
                raise RuntimeError("EE zip has no tif")
            data = zf.read(names[0])
    with rasterio.open(BytesIO(data)) as ds:
        arr = ds.read(out_shape=(len(bands), grid["n"], grid["n"]), resampling=Resampling.nearest).astype(np.float32)
    arr[arr <= NODATA / 2] = np.nan
    return arr

def landsat_image(row):
    return ee.Image(str(row["scene_id"]))

def landsat_band_map(row):
    scene_ref = " ".join(str(row.get(key, "")) for key in ["scene_id", "collection", "landsat_product_id", "selected_landsat_scene_id"])
    if "LT05" in scene_ref:
        return {
            "thermal": "ST_B6",
            "blue": "SR_B1",
            "red": "SR_B3",
            "nir": "SR_B4",
            "swir1": "SR_B5",
            "swir2": "SR_B7",
        }
    return {
        "thermal": "ST_B10",
        "blue": "SR_B2",
        "red": "SR_B4",
        "nir": "SR_B5",
        "swir1": "SR_B6",
        "swir2": "SR_B7",
    }

def qa_clear_mask(img):
    qa = img.select("QA_PIXEL")
    bad_bits = [0, 1, 2, 3, 4, 5]
    mask = ee.Image(1)
    for bit in bad_bits:
        mask = mask.And(qa.bitwiseAnd(1 << bit).eq(0))
    return mask


def landsat_lst_direct(row, grid):
    img = landsat_image(row)
    lst_band = landsat_band_map(row)["thermal"]
    lst = img.select(lst_band).multiply(0.00341802).add(149.0).rename("lst_k_raw")
    clear = qa_clear_mask(img).And(img.select(lst_band).gt(0)).rename("clear_mask")
    cloud = clear.Not().rename("cloud_fill_mask")
    stack = ee_stack(ee.Image.cat([lst.updateMask(clear), clear.toFloat(), cloud.toFloat()]), ["lst_k_raw", "clear_mask", "cloud_fill_mask"], grid)
    lst_raw = stack[0].astype(np.float32)
    clear_mask = (stack[1] > 0.5)
    cloud_fill_mask = (stack[2] > 0.5)
    invalid = ~np.isfinite(lst_raw) | (lst_raw < LST_VALID_MIN_K) | (lst_raw > LST_VALID_MAX_K)
    lst_raw[invalid] = np.nan
    clear_mask = clear_mask & np.isfinite(lst_raw)
    return lst_raw.astype(np.float32), clear_mask.astype(np.uint8), cloud_fill_mask.astype(np.uint8)

def terrain_direct(row, grid):
    dem = ee.Image("USGS/SRTMGL1_003").select("elevation").resample("bilinear")
    terrain = ee.Terrain.products(dem)
    slope = terrain.select("slope")
    aspect_rad = terrain.select("aspect").multiply(np.pi / 180.0)
    aspect_sin = aspect_rad.sin().rename("aspect_sin")
    aspect_cos = aspect_rad.cos().rename("aspect_cos")
    hillshade = ee.Terrain.hillshade(dem, float(row["sun_azimuth"]), float(row["sun_elevation"])).rename("hillshade_landsat")
    img = ee.Image.cat([dem.rename("elevation"), slope.rename("slope"), aspect_sin, aspect_cos, hillshade])
    return ee_stack(img, TERRAIN_BANDS, grid).astype(np.float32)

def fill_nan_by_expanding_mean_2d(arr, radii=ALBEDO_FILL_RADII_PX, fallback_value=ALBEDO_FALLBACK_VALUE):
    filled = np.asarray(arr, dtype=np.float32).copy()
    nan_before = int((~np.isfinite(filled)).sum())
    radius_used = np.zeros(filled.shape, dtype=np.int16)
    for radius in radii:
        pending = ~np.isfinite(filled)
        if not bool(pending.any()):
            break
        updates = []
        ys, xs = np.where(pending)
        for y, x in zip(ys, xs):
            y0, y1 = max(0, int(y) - int(radius)), min(filled.shape[0], int(y) + int(radius) + 1)
            x0, x1 = max(0, int(x) - int(radius)), min(filled.shape[1], int(x) + int(radius) + 1)
            window = filled[y0:y1, x0:x1]
            valid = np.isfinite(window)
            if bool(valid.any()):
                updates.append((int(y), int(x), float(np.nanmean(window[valid]))))
        for y, x, value in updates:
            filled[y, x] = value
            radius_used[y, x] = int(radius)
    pending = ~np.isfinite(filled)
    fallback_pixels = int(pending.sum())
    if fallback_pixels:
        fallback = float(np.nanmedian(filled)) if bool(np.isfinite(filled).any()) else float(fallback_value)
        filled[pending] = fallback
    nan_after = int((~np.isfinite(filled)).sum())
    return filled.astype(np.float32), {
        "nan_before": nan_before,
        "nan_after": nan_after,
        "local_fill_pixels": int(nan_before - fallback_pixels),
        "fallback_fill_pixels": fallback_pixels,
        "max_radius_px": int(radius_used.max()) if radius_used.size else 0,
    }


def fill_albedo_nan_stack(arr):
    filled = np.asarray(arr, dtype=np.float32).copy()
    per_band = []
    for band_idx in range(filled.shape[0]):
        filled[band_idx], stats = fill_nan_by_expanding_mean_2d(filled[band_idx])
        per_band.append(stats)
    return filled.astype(np.float32), {
        "albedo_fill_method": "attempt3_clear_mask_focal_mean_plus_local_expanding_mean",
        "albedo_nan_pixels_before": int(sum(item["nan_before"] for item in per_band)),
        "albedo_nan_pixels_after": int(sum(item["nan_after"] for item in per_band)),
        "albedo_local_fill_pixels": int(sum(item["local_fill_pixels"] for item in per_band)),
        "albedo_fallback_fill_pixels": int(sum(item["fallback_fill_pixels"] for item in per_band)),
        "albedo_fill_max_radius_px": int(max([item["max_radius_px"] for item in per_band] or [0])),
        "albedo_fallback_value": float(ALBEDO_FALLBACK_VALUE),
    }


def albedo_direct(row, grid):
    img = landsat_image(row)
    band_map = landsat_band_map(row)
    clear_img = img.updateMask(qa_clear_mask(img))
    def sr(name):
        return clear_img.select(name).multiply(0.0000275).add(-0.2)
    albedo = sr(band_map["blue"]).multiply(0.356).add(sr(band_map["red"]).multiply(0.130)).add(sr(band_map["nir"]).multiply(0.373)).add(sr(band_map["swir1"]).multiply(0.085)).add(sr(band_map["swir2"]).multiply(0.072)).subtract(0.0018).rename("albedo")
    # attempt3 생성법: clear pixel albedo를 만들고 focal mean으로 구름/결측을 먼저 보완한다.
    albedo_attempt3 = albedo.unmask(albedo.focal_mean(radius=5, units="pixels", iterations=2)).rename("albedo")
    # 해안/도서처럼 EE focal mean만으로 남는 빈칸은 넓은 focal mean과 로컬 확장 평균으로 끝까지 채운다.
    albedo_attempt3 = albedo_attempt3.unmask(albedo_attempt3.focal_mean(radius=16, units="pixels", iterations=1)).rename("albedo")
    arr = ee_stack(albedo_attempt3, ALBEDO_BANDS, grid).astype(np.float32)
    global albedo_fill_stats
    arr, albedo_fill_stats = fill_albedo_nan_stack(arr)
    return arr.astype(np.float32)

lst_k_raw_direct, lst_clear_mask_direct, lst_cloud_fill_mask_direct = landsat_lst_direct(row, grid)
terrain = terrain_direct(row, grid)
albedo = albedo_direct(row, grid)
print("lst raw:", lst_k_raw_direct.shape, "clear pixels:", int(lst_clear_mask_direct.sum()))
print("terrain:", terrain.shape, "albedo:", albedo.shape)
print("albedo fill stats:", albedo_fill_stats)


AWSH_BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/awsh.php"
STN_BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/stn_inf.php"
KMA_RETRIES = 5
KMA_BACKOFF_SECONDS = 3
AWSH_COLUMNS = {"TA": ["TM", "STN", "TA"], "WD": ["TM", "STN", "WD"], "WS": ["TM", "STN", "WS"], "RN": ["TM", "STN", "RN"], "HM": ["TM", "STN", "HM"]}
AWS_VALUE_RENAME = {"TA": "avg_temp", "RN": "rain_1h", "HM": "humidity"}

def decode_kma_response(content):
    for enc in ["utf-8", "cp949", "euc-kr"]:
        try:
            return content.decode(enc)
        except UnicodeDecodeError:
            continue
    return content.decode("utf-8", errors="replace")

def valid_data_lines(text):
    return [ln.strip() for ln in text.splitlines() if ln.strip() and not ln.startswith("#")]

def request_kma_with_retry(url, params, timeout=60):
    last_exc = None
    for attempt in range(1, KMA_RETRIES + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as exc:
            last_exc = exc
            wait = KMA_BACKOFF_SECONDS * attempt
            print(f"[KMA retry {attempt}/{KMA_RETRIES}] {params} -> {type(exc).__name__}: {exc}; wait {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"KMA request failed after retries: {params}; last={last_exc}")

def fetch_awsh_var(var, tm, auth_key):
    params = {"var": var, "tm": tm, "stn": "0", "help": "0", "authKey": auth_key}
    r = request_kma_with_retry(AWSH_BASE_URL, params, timeout=60)
    lines = valid_data_lines(decode_kma_response(r.content))
    rows = [re.split(r"\s+", ln)[: len(AWSH_COLUMNS[var])] for ln in lines]
    df = pd.DataFrame(rows, columns=AWSH_COLUMNS[var]) if rows else pd.DataFrame(columns=AWSH_COLUMNS[var])
    for col in df.columns:
        if col != "TM":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["STN"])
    df["STN"] = df["STN"].astype(int)
    return df

def parse_station_info_text(text):
    rows = []
    for ln in valid_data_lines(text):
        parts = re.split(r"\s+", ln)
        nums = [x for x in parts if re.fullmatch(r"-?\d+(\.\d+)?", x)]
        if len(nums) >= 3:
            stn = int(float(nums[0]))
            lon = float(nums[1])
            lat = float(nums[2])
            if 120.0 <= lon <= 135.0 and 30.0 <= lat <= 45.0:
                rows.append({"STN": stn, "LON": lon, "LAT": lat})
    return pd.DataFrame(rows, columns=["STN", "LON", "LAT"])


def fetch_station_info(auth_key, tm):
    # KMA가 stn="" 조합에서 500을 반환하는 경우가 있어 stn=0/help=0을 먼저 시도한다.
    variants = [
        {"inf": "AWS", "stn": "0", "tm": tm, "help": "0", "authKey": auth_key},
        {"inf": "AWS", "stn": "0", "tm": tm, "help": "1", "authKey": auth_key},
        {"inf": "AWS", "stn": "", "tm": tm, "help": "0", "authKey": auth_key},
        {"inf": "AWS", "stn": "", "tm": tm, "help": "1", "authKey": auth_key},
    ]
    errors = []
    for params in variants:
        try:
            r = request_kma_with_retry(STN_BASE_URL, params, timeout=60)
            text = decode_kma_response(r.content)
            df = parse_station_info_text(text)
            if not df.empty:
                print(f"[KMA station info] {len(df)} AWS stations via stn={params['stn']!r} help={params['help']}")
                return df.drop_duplicates("STN")
            errors.append(f"stn={params['stn']!r} help={params['help']}: empty parse")
        except Exception as exc:
            errors.append(f"stn={params['stn']!r} help={params['help']}: {type(exc).__name__}: {exc}")
    raise RuntimeError("KMA station info returned no AWS stations. Tried variants: " + " | ".join(errors))


def fetch_aws_frame_direct(date_ymd, auth_key):
    tm = pd.Timestamp(date_ymd).strftime("%Y%m%d") + HOUR_KST
    aws = fetch_station_info(auth_key, tm)
    for var in AWSH_COLUMNS:
        df = fetch_awsh_var(var, tm, auth_key)
        aws = aws.merge(df[["STN", var]], on="STN", how="left")
    aws = aws.rename(columns=AWS_VALUE_RENAME)
    wd = np.deg2rad(aws["WD"].to_numpy(dtype=np.float32))
    ws = aws["WS"].to_numpy(dtype=np.float32)
    aws["wind_u"] = -ws * np.sin(wd)
    aws["wind_v"] = -ws * np.cos(wd)
    keep = ["STN", "LON", "LAT", "avg_temp", "wind_u", "wind_v", "rain_1h", "humidity"]
    aws = aws[keep].dropna(subset=["LON", "LAT"])
    if aws.empty:
        raise RuntimeError(f"KMA AWS frame has no station coordinates for {tm}")
    return aws, tm

def idw_grid(points_xy, values, grid, power=2.0, k=12):
    pts = np.asarray(points_xy, dtype=np.float32)
    vals = np.asarray(values, dtype=np.float32)
    valid = np.isfinite(vals)
    pts, vals = pts[valid], vals[valid]
    if len(vals) == 0:
        return np.full((grid["n"], grid["n"]), np.nan, dtype=np.float32)
    out = np.empty((grid["n"], grid["n"]), dtype=np.float32)
    flat_x, flat_y, out_flat = grid["xx"].ravel(), grid["yy"].ravel(), out.ravel()
    for start in range(0, flat_x.size, 4096):
        end = min(start + 4096, flat_x.size)
        dx = flat_x[start:end, None] - pts[None, :, 0]
        dy = flat_y[start:end, None] - pts[None, :, 1]
        dist = np.sqrt(dx * dx + dy * dy)
        kk = min(k, dist.shape[1])
        idx = np.argpartition(dist, kk - 1, axis=1)[:, :kk]
        d = np.take_along_axis(dist, idx, axis=1)
        v = vals[idx]
        w = 1.0 / np.maximum(d, 1e-6) ** power
        pred = np.sum(w * v, axis=1) / np.sum(w, axis=1)
        exact = d < 1e-6
        if exact.any():
            rows = np.where(exact.any(axis=1))[0]
            for rr in rows:
                pred[rr] = v[rr, np.argmax(exact[rr])]
        out_flat[start:end] = pred.astype(np.float32)
    return out

def aws_patch_direct(date_ymd, grid):
    aws, tm = fetch_aws_frame_direct(date_ymd, KMA_AUTH_KEY)
    to_xy = Transformer.from_crs("EPSG:4326", grid["crs"], always_xy=True)
    xs, ys = to_xy.transform(aws["LON"].to_numpy(), aws["LAT"].to_numpy())
    pts = np.column_stack([xs, ys])
    stacks = []
    for band in AWS_BANDS:
        stacks.append(idw_grid(pts, aws[band].to_numpy(dtype=np.float32), grid)[None])
    return np.concatenate(stacks, axis=0).astype(np.float32), aws, tm

aws, aws_station_frame, aws_tm = aws_patch_direct(landsat_info["selected_landsat_date"], grid)
print("aws:", aws.shape, "tm:", aws_tm, "stations:", len(aws_station_frame))


EGIS_WMS_URL = "https://api.mcee.go.kr/geoserver/wms?"
EGIS_COLOR_MATCH_VERSION = "rgbtol_strict10_supersample4_marg8_announced_v1"
WMS_SUPERSAMPLE = 4
LV1_CLASSES = [100, 200, 300, 400, 500, 600, 700]
URBAN_LV2_CLASSES = [110, 120, 130, 140, 150, 160]
WATER_LV2_CLASSES = [710, 720]
OTHER_LV1_CLASSES = [200, 300, 400, 500, 600]
HYBRID_CLASSES = URBAN_LV2_CLASSES + OTHER_LV1_CLASSES + WATER_LV2_CLASSES
HYBRID_CLASS_NAMES = {110: "urban_residential", 120: "urban_industrial", 130: "urban_commercial", 140: "urban_cultural_public", 150: "urban_transportation", 160: "urban_mixed", 200: "agriculture", 300: "forest", 400: "grassland", 500: "wetland", 600: "barren", 710: "water_inland", 720: "water_marine"}
HYBRID_FILL_PRIORITY = {cls: i for i, cls in enumerate([720, 710, 110, 120, 130, 140, 150, 160, 300, 200, 400, 500, 600])}
LV1_COLORS = {100: (255, 0, 0), 200: (238, 233, 7), 300: (42, 75, 45), 400: (57, 150, 38), 500: (124, 34, 126), 600: (89, 206, 202), 700: (6, 2, 250)}
LV2_COLORS = {110: (254, 230, 194), 120: (192, 132, 132), 130: (237, 131, 184), 140: (246, 113, 138), 150: (247, 65, 42), 160: (246, 177, 18), 210: (255, 255, 191), 220: (247, 249, 102), 230: (223, 220, 115), 240: (184, 177, 44), 250: (184, 145, 18), 310: (51, 160, 44), 320: (10, 79, 64), 330: (51, 102, 51), 410: (161, 213, 148), 420: (96, 126, 51), 510: (180, 167, 208), 520: (153, 116, 153), 610: (193, 219, 236), 620: (159, 242, 255), 710: (62, 167, 255), 720: (23, 57, 255)}
LV3_COLORS = {111: (254, 230, 194), 112: (223, 193, 111), 121: (192, 132, 132), 131: (237, 131, 184), 132: (223, 176, 164), 141: (246, 113, 138), 151: (229, 38, 254), 152: (197, 50, 81), 153: (252, 4, 78), 154: (247, 65, 42), 155: (115, 0, 0), 161: (246, 177, 18), 162: (255, 122, 0), 163: (199, 88, 27), 211: (255, 255, 191), 212: (244, 230, 168), 221: (247, 249, 102), 222: (245, 228, 10), 231: (223, 220, 115), 241: (184, 177, 44), 251: (184, 145, 18), 252: (170, 100, 0), 311: (51, 160, 44), 321: (10, 79, 64), 331: (51, 102, 51), 411: (161, 213, 148), 421: (128, 228, 90), 422: (113, 176, 90), 423: (96, 126, 51), 511: (180, 167, 208), 521: (153, 116, 153), 522: (124, 30, 162), 611: (193, 219, 236), 612: (171, 197, 202), 613: (171, 182, 165), 621: (88, 90, 138), 622: (123, 181, 172), 623: (159, 242, 255), 711: (62, 167, 255), 712: (93, 109, 255), 721: (23, 57, 255)}
COLOR_TABLE_BY_LEVEL = {"lv1": LV1_COLORS, "lv2": LV2_COLORS, "lv3": LV3_COLORS}
WMS_COLOR_DISTANCE_THRESHOLD_BY_LEVEL = {"lv1": 18.0, "lv2": 18.0, "lv3": 18.0}
WMS_COLOR_CHANNEL_TOLERANCE_BY_LEVEL = {"lv1": 10.0, "lv2": 10.0, "lv3": 10.0}
WMS_COLOR_MARGIN_MIN_BY_LEVEL = {"lv1": 8.0, "lv2": 8.0, "lv3": 8.0}
FILL_CLASS_BY_CHOICE = {"sea": 720, "urban": 110}
EGIS_REFERENCE_FILL_MAX_YEAR = 2021
EGIS_REFERENCE_YEAR = 2022
EGIS_REFERENCE_FILL_WEIGHT = 4
EGIS_MAX_FILL_RADIUS_PX = 16
EGIS_PIXEL_SIZE_M = 60.0

def egis_policy_for_year(year):
    year = int(year)
    if year <= 2015:
        return {"policy_year": 2007, "source_key": "lv2_2007_g", "source_level": "lv2", "layer": "EGIS:lv2_2007_g"}
    rows = [
        (2016, 2018, 2018, "lv3_10th_g", "lv3"),
        (2019, 2019, 2019, "lv3_2020_g", "lv3"),
        (2020, 2020, 2020, "lv3_2021_g", "lv3"),
        (2021, 2021, 2021, "lv2_2022y", "lv2"),
        (2022, 2022, 2022, "lv2_2023y", "lv2"),
        (2023, 2023, 2023, "lv2_2024y", "lv2"),
        (2024, 2026, 2024, "lv2_2025y", "lv2"),
    ]
    for amin, amax, py, key, level in rows:
        if amin <= year <= amax:
            return {"policy_year": py, "source_key": key, "source_level": level, "layer": f"EGIS:{key}"}
    raise ValueError(f"unsupported EGIS analysis year: {year}")

def code_to_lv1(code):
    code = int(code)
    if code <= 0:
        return 0
    if code in LV1_CLASSES:
        return code
    return (code // 100) * 100

def code_to_lv2(code, level="lv3"):
    code = int(code)
    if code <= 0:
        return 0
    if level in {"lv1", "lv2"}:
        return code
    return (code // 10) * 10

def code_to_hybrid(code, level="lv3"):
    code = int(code)
    if code <= 0:
        return 0
    lv1 = code_to_lv1(code)
    lv2 = code_to_lv2(code, level)
    if lv1 == 100:
        return lv2 if lv2 in URBAN_LV2_CLASSES else 110
    if lv1 == 700:
        return lv2 if lv2 in WATER_LV2_CLASSES else 720
    if lv1 in OTHER_LV1_CLASSES:
        return lv1
    return 0

def request_wms_image(layer_name, grid, scale=1):
    width = int(grid["n"] * scale)
    height = int(grid["n"] * scale)
    params = {"SERVICE": "WMS", "VERSION": "1.1.1", "REQUEST": "GetMap", "LAYERS": layer_name, "STYLES": "", "SRS": grid["crs"], "BBOX": f'{grid["left"]},{grid["bottom"]},{grid["right"]},{grid["top"]}', "WIDTH": width, "HEIGHT": height, "FORMAT": "image/png", "TRANSPARENT": "TRUE"}
    last_exc = None
    for attempt in range(1, 6):
        try:
            r = requests.get(EGIS_WMS_URL, params=params, timeout=REQUEST_TIMEOUT)
            content_type = r.headers.get("content-type", "")
            if not r.ok:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")
            if "image" not in content_type.lower() and not r.content.startswith(bytes([137, 80, 78, 71])):
                raise RuntimeError(f"WMS did not return image: {content_type}; {r.text[:500]}")
            img = Image.open(BytesIO(r.content)).convert("RGBA")
            if img.size != (width, height):
                img = img.resize((width, height), Image.Resampling.NEAREST)
            return np.array(img)
        except Exception as exc:
            last_exc = exc
            wait = min(5 * attempt, 30)
            print(f"[EGIS WMS retry {attempt}/5] {layer_name}: {type(exc).__name__}: {exc}; wait {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"EGIS WMS failed for {layer_name}: {last_exc}")

def rgb_to_hybrid_landcover(rgba, level):
    rgb = rgba[..., :3].astype(np.float32).reshape(-1, 3)
    alpha = rgba[..., 3].reshape(-1)
    table = COLOR_TABLE_BY_LEVEL[level]
    codes = np.array(list(table.keys()), dtype=np.int16)
    colors = np.array([table[int(c)] for c in codes], dtype=np.float32)
    raw_flat = np.zeros(rgb.shape[0], dtype=np.int16)
    accepted_flat = np.zeros(rgb.shape[0], dtype=np.uint8)
    dist_flat = np.full(rgb.shape[0], np.inf, dtype=np.float32)
    absmax_flat = np.full(rgb.shape[0], np.inf, dtype=np.float32)
    dist_thr = WMS_COLOR_DISTANCE_THRESHOLD_BY_LEVEL[level]
    ch_thr = WMS_COLOR_CHANNEL_TOLERANCE_BY_LEVEL[level]
    margin_thr = WMS_COLOR_MARGIN_MIN_BY_LEVEL[level]
    for start in range(0, rgb.shape[0], 200_000):
        end = min(start + 200_000, rgb.shape[0])
        chunk = rgb[start:end]
        diff = chunk[:, None, :] - colors[None, :, :]
        dist_sq = np.sum(diff * diff, axis=-1)
        best_idx = np.argmin(dist_sq, axis=-1)
        best_dist = np.sqrt(dist_sq[np.arange(end - start), best_idx])
        second_sq = np.partition(dist_sq, 1, axis=-1)[:, 1] if dist_sq.shape[1] > 1 else np.full(end - start, np.inf)
        margin = np.sqrt(second_sq) - best_dist
        best_absmax = np.max(np.abs(diff[np.arange(end - start), best_idx, :]), axis=-1)
        accepted = (alpha[start:end] >= 128) & (best_dist <= dist_thr) & (best_absmax <= ch_thr) & (margin >= margin_thr)
        raw_flat[start:end] = codes[best_idx]
        accepted_flat[start:end] = accepted.astype(np.uint8)
        dist_flat[start:end] = best_dist.astype(np.float32)
        absmax_flat[start:end] = best_absmax.astype(np.float32)
    hybrid_flat = np.zeros_like(raw_flat)
    raw_to_hybrid = {int(code): int(code_to_hybrid(int(code), level)) for code in codes}
    idx = np.where(accepted_flat.astype(bool))[0]
    hybrid_flat[idx] = np.array([raw_to_hybrid[int(c)] for c in raw_flat[idx]], dtype=np.int16)
    shape = rgba.shape[:2]
    source_nodata = (rgba[..., 3] < 128)
    unrecognized = (~source_nodata) & (accepted_flat.reshape(shape) == 0)
    return hybrid_flat.reshape(shape), raw_flat.reshape(shape), source_nodata.astype(np.uint8), unrecognized.astype(np.uint8), dist_flat.reshape(shape), absmax_flat.reshape(shape)

def downsample_rgba_preview(rgba, out_n):
    return np.array(Image.fromarray(rgba.astype(np.uint8), mode="RGBA").resize((out_n, out_n), Image.Resampling.NEAREST), dtype=np.uint8)

def block_majority_bool(arr, factor):
    h, w = arr.shape
    blocks = arr[: h // factor * factor, : w // factor * factor].reshape(h // factor, factor, w // factor, factor).transpose(0, 2, 1, 3).reshape(h // factor, w // factor, factor * factor)
    return (blocks.mean(axis=-1) >= 0.5).astype(np.uint8)

def block_mean_float(arr, factor):
    h, w = arr.shape
    blocks = arr[: h // factor * factor, : w // factor * factor].reshape(h // factor, factor, w // factor, factor).transpose(0, 2, 1, 3).reshape(h // factor, w // factor, factor * factor)
    return np.nanmean(blocks.astype(np.float32), axis=-1).astype(np.float32)

def block_max_float(arr, factor):
    h, w = arr.shape
    blocks = arr[: h // factor * factor, : w // factor * factor].reshape(h // factor, factor, w // factor, factor).transpose(0, 2, 1, 3).reshape(h // factor, w // factor, factor * factor)
    return np.nanmax(blocks.astype(np.float32), axis=-1).astype(np.float32)

def block_mode_codes(arr, factor, classes, priority=None):
    h, w = arr.shape
    blocks = arr[: h // factor * factor, : w // factor * factor].reshape(h // factor, factor, w // factor, factor).transpose(0, 2, 1, 3).reshape(h // factor, w // factor, factor * factor)
    out = np.zeros(blocks.shape[:2], dtype=arr.dtype)
    best_count = np.zeros(blocks.shape[:2], dtype=np.int16)
    ordered = sorted([int(c) for c in classes], key=lambda c: (priority or {}).get(c, 9999))
    for cls in ordered:
        counts = (blocks == int(cls)).sum(axis=-1).astype(np.int16)
        take = counts > best_count
        out[take] = int(cls)
        best_count[take] = counts[take]
    return out

def egis_direct_for_policy(policy, grid):
    rgba_hr = request_wms_image(policy["layer"], grid, scale=WMS_SUPERSAMPLE)
    hybrid_hr, raw_code_hr, source_nodata_hr, unrecognized_hr, dist_hr, absmax_hr = rgb_to_hybrid_landcover(rgba_hr, policy["source_level"])
    priority = {0: 9999, **HYBRID_FILL_PRIORITY}
    raw_classes = [0] + [int(c) for c in COLOR_TABLE_BY_LEVEL[policy["source_level"]].keys()]
    major = block_mode_codes(hybrid_hr.astype(np.int16), WMS_SUPERSAMPLE, [0] + HYBRID_CLASSES, priority)
    raw_code = block_mode_codes(raw_code_hr.astype(np.int16), WMS_SUPERSAMPLE, raw_classes, {0: 9999})
    source_nodata = block_majority_bool(source_nodata_hr, WMS_SUPERSAMPLE)
    unrecognized = block_majority_bool(unrecognized_hr, WMS_SUPERSAMPLE)
    rgba = downsample_rgba_preview(rgba_hr, grid["n"])
    color_diag = {
        "egis_color_match_version": np.array(EGIS_COLOR_MATCH_VERSION),
        "egis_wms_supersample": np.array(int(WMS_SUPERSAMPLE)),
        "egis_rgb_distance_threshold": np.array(float(WMS_COLOR_DISTANCE_THRESHOLD_BY_LEVEL[policy["source_level"]]), dtype=np.float32),
        "egis_rgb_channel_tolerance": np.array(float(WMS_COLOR_CHANNEL_TOLERANCE_BY_LEVEL[policy["source_level"]]), dtype=np.float32),
        "egis_rgb_margin_min": np.array(float(WMS_COLOR_MARGIN_MIN_BY_LEVEL[policy["source_level"]]), dtype=np.float32),
        "egis_rgb_distance_mean_60m": block_mean_float(dist_hr, WMS_SUPERSAMPLE),
        "egis_rgb_distance_max_60m": block_max_float(dist_hr, WMS_SUPERSAMPLE),
        "egis_rgb_absmax_mean_60m": block_mean_float(absmax_hr, WMS_SUPERSAMPLE),
        "egis_rgb_absmax_max_60m": block_max_float(absmax_hr, WMS_SUPERSAMPLE),
        "egis_unrecognized_fraction_60m": block_mean_float(unrecognized_hr.astype(np.float32), WMS_SUPERSAMPLE),
        "egis_source_nodata_fraction_60m": block_mean_float(source_nodata_hr.astype(np.float32), WMS_SUPERSAMPLE),
    }
    major[source_nodata.astype(bool) | unrecognized.astype(bool)] = 0
    return policy, rgba, major.astype(np.int16), raw_code.astype(np.int16), source_nodata, unrecognized, color_diag

def egis_direct(row, grid):
    policy = egis_policy_for_year(pd.Timestamp(row["date"]).year)
    return egis_direct_for_policy(policy, grid)

egis_policy, egis_rgba, egis_major, egis_raw_code, egis_source_nodata, egis_unrecognized, egis_color_diag = egis_direct(row, grid)
raw_egis_npz = DIRECT_RAW_DIR / f"egis_direct_raw_{RUN_LABEL}_{egis_policy['source_key']}.npz"
np.savez_compressed(raw_egis_npz, raw_rgba=egis_rgba, egis_hybrid_lc_unfilled_60m=egis_major, egis_raw_code_60m=egis_raw_code, egis_source_nodata_mask=egis_source_nodata, egis_unrecognized_rgb_mask=egis_unrecognized, **egis_color_diag, **{k: np.array(v) for k, v in egis_policy.items()})
print("EGIS direct:", egis_policy, "raw:", raw_egis_npz)


def hybrid_to_rgba(arr):
    out = np.zeros((*arr.shape, 4), dtype=np.uint8)
    colors = {**LV1_COLORS, **LV2_COLORS}
    for cls in HYBRID_CLASSES:
        m = arr == int(cls)
        out[m, :3] = np.array(colors.get(int(cls), (0, 0, 0)), dtype=np.uint8)
        out[m, 3] = 255
    out[arr == 0] = np.array([0, 0, 0, 255], dtype=np.uint8)
    return out

def mask_to_rgba(mask, on=(255, 0, 255), off=(0, 0, 0)):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    rgb[~mask.astype(bool)] = np.array(off, dtype=np.uint8)
    rgb[mask.astype(bool)] = np.array(on, dtype=np.uint8)
    return np.dstack([rgb, np.full(mask.shape, 255, dtype=np.uint8)])

def add_label(img, label, height=24):
    base = Image.new("RGBA", (img.width, img.height + height), (255, 255, 255, 255))
    base.paste(img, (0, height))
    draw = ImageDraw.Draw(base)
    draw.text((4, 5), str(label)[:80], fill=(0, 0, 0, 255))
    return base

def save_sheet(items, out_path):
    labeled = [add_label(Image.fromarray(arr).convert("RGBA").resize((256, 256), Image.Resampling.NEAREST), title) for title, arr in items]
    cols = 3
    rows = int(math.ceil(len(labeled) / cols))
    sheet = Image.new("RGBA", (cols * 256, rows * 280), (255, 255, 255, 255))
    for i, img in enumerate(labeled):
        sheet.paste(img, ((i % cols) * 256, (i // cols) * 280))
    sheet.save(out_path)

def choose_weighted_majority(counts):
    best_cls = 0
    best_count = 0
    best_priority = 9999
    for cls in HYBRID_CLASSES:
        count = int(counts[int(cls)]) if int(cls) < len(counts) else 0
        priority = HYBRID_FILL_PRIORITY.get(int(cls), 9999)
        if count > best_count or (count == best_count and count > 0 and priority < best_priority):
            best_cls = int(cls)
            best_count = count
            best_priority = priority
    return best_cls

def neighbor_counts(filled, y, x):
    y0, y1 = max(0, y - 1), min(filled.shape[0], y + 2)
    x0, x1 = max(0, x - 1), min(filled.shape[1], x + 2)
    vals = filled[y0:y1, x0:x1].reshape(-1)
    vals = vals[vals > 0]
    counts = np.zeros(max(HYBRID_CLASSES) + 1, dtype=np.int16)
    if vals.size:
        uniq, cnt = np.unique(vals.astype(np.int16), return_counts=True)
        counts[uniq] += cnt.astype(np.int16)
    return counts

def fill_by_iterative_neighbors(major, target_mask, reference_lc=None, reference_weight=0, max_radius_px=EGIS_MAX_FILL_RADIUS_PX):
    filled = major.copy().astype(np.int16)
    pending = target_mask.astype(bool).copy()
    fill_mask = np.zeros_like(pending, dtype=bool)
    reference_vote_mask = np.zeros_like(pending, dtype=bool)
    radius_used = np.zeros(filled.shape, dtype=np.int16)
    for radius in range(1, int(max_radius_px) + 1):
        ys, xs = np.where(pending)
        if len(ys) == 0:
            break
        updates = []
        used_reference = []
        for y, x in zip(ys, xs):
            counts = neighbor_counts(filled, int(y), int(x))
            ref_used = False
            if reference_lc is not None:
                ref_cls = int(reference_lc[int(y), int(x)])
                if ref_cls > 0:
                    counts[ref_cls] += int(reference_weight)
                    ref_used = True
            winner = choose_weighted_majority(counts)
            if winner > 0:
                updates.append((int(y), int(x), int(winner)))
                used_reference.append(ref_used)
        if not updates:
            break
        for (y, x, winner), ref_used in zip(updates, used_reference):
            filled[y, x] = winner
            pending[y, x] = False
            fill_mask[y, x] = True
            reference_vote_mask[y, x] = bool(ref_used)
            radius_used[y, x] = radius
    return filled, fill_mask, reference_vote_mask, pending, radius_used

def fill_by_expanding_window(major, target_mask, max_radius_px=EGIS_MAX_FILL_RADIUS_PX):
    filled = major.copy().astype(np.int16)
    pending = target_mask.astype(bool).copy()
    fill_mask = np.zeros_like(pending, dtype=bool)
    radius_used = np.zeros(filled.shape, dtype=np.int16)
    for radius in range(1, int(max_radius_px) + 1):
        ys, xs = np.where(pending)
        if len(ys) == 0:
            break
        updates = []
        for y, x in zip(ys, xs):
            y0, y1 = max(0, int(y) - radius), min(filled.shape[0], int(y) + radius + 1)
            x0, x1 = max(0, int(x) - radius), min(filled.shape[1], int(x) + radius + 1)
            vals = filled[y0:y1, x0:x1].reshape(-1)
            vals = vals[vals > 0]
            if vals.size == 0:
                continue
            counts = np.zeros(max(HYBRID_CLASSES) + 1, dtype=np.int16)
            uniq, cnt = np.unique(vals.astype(np.int16), return_counts=True)
            counts[uniq] += cnt.astype(np.int16)
            winner = choose_weighted_majority(counts)
            if winner > 0:
                updates.append((int(y), int(x), int(winner)))
        if not updates:
            continue
        for y, x, winner in updates:
            filled[y, x] = winner
            pending[y, x] = False
            fill_mask[y, x] = True
            radius_used[y, x] = radius
    return filled, fill_mask, pending, radius_used

def load_reference_egis_lc_for_fill():
    ref_policy = egis_policy_for_year(EGIS_REFERENCE_YEAR)
    ref_policy, ref_rgba, ref_major, ref_raw_code, ref_source_nodata, ref_unrecognized, ref_color_diag = egis_direct_for_policy(ref_policy, grid)
    ref_unknown = (ref_major == 0) | ref_source_nodata.astype(bool) | ref_unrecognized.astype(bool)
    if bool(ref_unknown.any()):
        ref_major, ref_majority_mask, ref_leftover, ref_radius = fill_by_expanding_window(ref_major, ref_unknown, EGIS_MAX_FILL_RADIUS_PX)
        if bool(ref_leftover.any()):
            print(f"2022 reference still has {int(ref_leftover.sum())} unknown pixels after radius cap; ignored as reference votes.")
            ref_major[ref_leftover] = 0
    ref_npz = DIRECT_RAW_DIR / f"egis_direct_raw_{RUN_LABEL}_reference_{ref_policy['source_key']}.npz"
    np.savez_compressed(
        ref_npz,
        raw_rgba=ref_rgba,
        egis_hybrid_lc_ref_filled_60m=ref_major.astype(np.int16),
        egis_raw_code_60m=ref_raw_code.astype(np.int16),
        egis_source_nodata_mask=ref_source_nodata.astype(np.uint8),
        egis_unrecognized_rgb_mask=ref_unrecognized.astype(np.uint8),
        reference_year=np.array(int(EGIS_REFERENCE_YEAR)),
        reference_fill_radius_cap_px=np.array(int(EGIS_MAX_FILL_RADIUS_PX)),
        **{f"reference_{k}": v for k, v in ref_color_diag.items()},
        **{k: np.array(v) for k, v in ref_policy.items()},
    )
    return ref_major.astype(np.int16), ref_npz

def auto_fill_mask(major, target_mask, analysis_year, stats_prefix):
    stats = {
        f"{stats_prefix}_fill_method": "none",
        f"{stats_prefix}_input_pixels": int(target_mask.sum()),
        f"{stats_prefix}_reference_fill_year": -1,
        f"{stats_prefix}_reference_fill_weight": 0,
        f"{stats_prefix}_reference_vote_pixels": 0,
        f"{stats_prefix}_majority_fill_pixels": 0,
        f"{stats_prefix}_fill_max_radius_px": 0,
        f"{stats_prefix}_fill_max_radius_m": 0.0,
        f"{stats_prefix}_fill_radius_cap_px": int(EGIS_MAX_FILL_RADIUS_PX),
        f"{stats_prefix}_fill_radius_cap_m": float(EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M),
        f"{stats_prefix}_leftover_pixels": 0,
        f"{stats_prefix}_reference_raw_npz": "",
    }
    if not bool(target_mask.any()):
        return major.copy().astype(np.int16), stats
    reference_lc = None
    reference_weight = 0
    if int(analysis_year) <= EGIS_REFERENCE_FILL_MAX_YEAR:
        reference_lc, ref_npz = load_reference_egis_lc_for_fill()
        reference_weight = int(EGIS_REFERENCE_FILL_WEIGHT)
        stats.update({
            f"{stats_prefix}_fill_method": "attempt3_8nbr_plus_2022x4",
            f"{stats_prefix}_reference_fill_year": int(EGIS_REFERENCE_YEAR),
            f"{stats_prefix}_reference_fill_weight": int(EGIS_REFERENCE_FILL_WEIGHT),
            f"{stats_prefix}_reference_raw_npz": str(ref_npz),
        })
    else:
        stats[f"{stats_prefix}_fill_method"] = "attempt3_expanding_neighbor_majority"
    filled, neighbor_mask, ref_vote_mask, pending, radius_used = fill_by_iterative_neighbors(
        major,
        target_mask,
        reference_lc=reference_lc,
        reference_weight=reference_weight,
        max_radius_px=EGIS_MAX_FILL_RADIUS_PX,
    )
    if bool(pending.any()):
        filled, window_mask, pending, window_radius = fill_by_expanding_window(filled, pending, EGIS_MAX_FILL_RADIUS_PX)
        radius_used = np.maximum(radius_used, window_radius)
        neighbor_mask = neighbor_mask | window_mask
        stats[f"{stats_prefix}_fill_method"] += "_plus_bounded_window_majority"
    stats[f"{stats_prefix}_reference_vote_pixels"] = int(ref_vote_mask.sum())
    stats[f"{stats_prefix}_majority_fill_pixels"] = int(neighbor_mask.sum())
    stats[f"{stats_prefix}_leftover_pixels"] = int(pending.sum())
    stats[f"{stats_prefix}_fill_max_radius_px"] = int(radius_used.max()) if radius_used.size else 0
    stats[f"{stats_prefix}_fill_max_radius_m"] = float(stats[f"{stats_prefix}_fill_max_radius_px"] * EGIS_PIXEL_SIZE_M)
    if bool(pending.any()):
        raise RuntimeError(
            f"EGIS {stats_prefix} fill left {int(pending.sum())} pixels after {EGIS_MAX_FILL_RADIUS_PX}px "
            f"({EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M:.0f}m) cap."
        )
    return filled.astype(np.int16), stats

# Step 1: resolve unrecognized RGB first. User choice is applied only after this step.
egis_major_raw = egis_major.copy().astype(np.int16)
egis_unrecognized_mask = egis_unrecognized.astype(bool) & (egis_major_raw == 0)
egis_major, egis_unrecognized_fill_stats = auto_fill_mask(
    egis_major_raw,
    egis_unrecognized_mask,
    analysis_year,
    "egis_unrecognized",
)
unrecognized_filled_npz = DIRECT_RAW_DIR / f"egis_unrecognized_filled_{RUN_LABEL}_{egis_policy['source_key']}.npz"
np.savez_compressed(
    unrecognized_filled_npz,
    egis_hybrid_lc_before_unrecognized_fill_60m=egis_major_raw.astype(np.int16),
    egis_hybrid_lc_after_unrecognized_fill_60m=egis_major.astype(np.int16),
    egis_unrecognized_rgb_mask=egis_unrecognized.astype(np.uint8),
    egis_source_nodata_mask=egis_source_nodata.astype(np.uint8),
    **{k: np.array(v) for k, v in egis_unrecognized_fill_stats.items()},
)
egis_unrecognized_fill_stats["egis_unrecognized_filled_npz"] = str(unrecognized_filled_npz)
print("EGIS unrecognized fill stats:", egis_unrecognized_fill_stats)

def save_unknown_review():
    # attempt3 policy: show remaining source nodata/zero pixels, then fill them automatically by majority/reference.
    unknown = ((egis_source_nodata.astype(bool) | (egis_major == 0)) & (egis_major == 0))
    overlay = egis_rgba[..., :3].copy()
    overlay[unknown] = np.array([255, 0, 255], dtype=np.uint8)
    overlay = np.dstack([overlay, np.full(unknown.shape, 255, dtype=np.uint8)])
    png = REVIEW_DIR / f"unknown_{RUN_LABEL}_{egis_policy['source_key']}.png"
    save_sheet([
        ("raw EGIS RGBA", egis_rgba),
        ("hybrid before unrecognized fill", hybrid_to_rgba(egis_major_raw)),
        (f"unrecognized filled={int(egis_unrecognized_mask.sum())}", mask_to_rgba(egis_unrecognized_mask, on=(255, 128, 0))),
        ("hybrid after unrecognized fill", hybrid_to_rgba(egis_major)),
        (f"unknown for auto fill={int(unknown.sum())}", mask_to_rgba(unknown)),
        (f"source_nodata={int(egis_source_nodata.sum())}", mask_to_rgba(egis_source_nodata.astype(bool), on=(255, 255, 255))),
    ], png)
    row_out = {
        "run_label": RUN_LABEL,
        "roi_label": ROI_LABEL,
        "requested_date": landsat_info["requested_date"],
        "selected_landsat_date": landsat_info["selected_landsat_date"],
        "roi_center_lat": float(ROI_CENTER_LAT),
        "roi_center_lon": float(ROI_CENTER_LON),
        "policy_year": int(egis_policy["policy_year"]),
        "source_key": egis_policy["source_key"],
        "source_level": egis_policy["source_level"],
        "raw_direct_egis_npz": str(raw_egis_npz),
        "unrecognized_filled_npz": str(unrecognized_filled_npz),
        "unrecognized_pixels": int(egis_unrecognized_mask.sum()),
        "unknown_pixels": int(unknown.sum()),
        "unknown_ratio": float(unknown.mean()),
        "preview_png": str(png),
        "fill_choice": "auto_attempt3",
        "fill_class": -1,
    }
    row_out.update(egis_unrecognized_fill_stats)
    pd.DataFrame([row_out]).to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")
    pd.DataFrame([row_out]).to_csv(DECISION_CSV, index=False, encoding="utf-8-sig")
    return unknown, png

unknown_mask, preview_png = save_unknown_review()
print("unknown review png:", preview_png)
print("unknown pixels for automatic fill:", int(unknown_mask.sum()), "ratio:", float(unknown_mask.mean()))
print("decision csv:", DECISION_CSV)


def read_fill_choice():
    # attempt3와 동일하게 바다/도심 수동 강제 없이 남은 source_nodata/zero만 자동 보간한다.
    if not bool(unknown_mask.any()):
        return "none", None
    print("EGIS unknown fill selected: auto_attempt3 (no sea/urban forced fill)")
    return "auto", None

def record_fill_decision(choice, cls, stats):
    if not DECISION_CSV.exists():
        return
    df = pd.read_csv(DECISION_CSV)
    if df.empty:
        return
    df.loc[0, "fill_choice"] = "auto_attempt3" if choice == "auto" else choice
    df.loc[0, "fill_class"] = -1
    for key, value in stats.items():
        if isinstance(value, (np.integer, np.floating)):
            value = value.item()
        df.loc[0, key] = value
    df.to_csv(DECISION_CSV, index=False, encoding="utf-8-sig")

def egis_onehot_from_lc(lc):
    if (lc == 0).any():
        raise RuntimeError("EGIS landcover still has unknown pixels")
    arrays, names = [], []
    for cls in HYBRID_CLASSES:
        label = HYBRID_CLASS_NAMES[int(cls)]
        arrays.append((lc == int(cls)).astype(np.float32)[None])
        names.append(f"egis_cat_{int(cls)}_{label}")
    return np.concatenate(arrays, axis=0), names

def sample_meta(date_ymd, sun_azimuth, sun_elevation, lat, lon):
    doy = pd.Timestamp(date_ymd).dayofyear
    return np.array([
        math.sin(2 * math.pi * doy / 366.0),
        math.cos(2 * math.pi * doy / 366.0),
        float(sun_azimuth) / 360.0,
        float(sun_elevation) / 90.0,
        (float(lat) - LAT_META_MEAN) / LAT_META_STD,
        (float(lon) - LON_META_MEAN) / LON_META_STD,
    ], dtype=np.float32)

def choose_weighted_majority(counts):
    best_cls = 0
    best_count = 0
    best_priority = 9999
    for cls in HYBRID_CLASSES:
        count = int(counts[int(cls)]) if int(cls) < len(counts) else 0
        priority = HYBRID_FILL_PRIORITY.get(int(cls), 9999)
        if count > best_count or (count == best_count and count > 0 and priority < best_priority):
            best_cls = int(cls)
            best_count = count
            best_priority = priority
    return best_cls

def neighbor_counts(filled, y, x):
    y0, y1 = max(0, y - 1), min(filled.shape[0], y + 2)
    x0, x1 = max(0, x - 1), min(filled.shape[1], x + 2)
    vals = filled[y0:y1, x0:x1].reshape(-1)
    vals = vals[vals > 0]
    counts = np.zeros(max(HYBRID_CLASSES) + 1, dtype=np.int16)
    if vals.size:
        uniq, cnt = np.unique(vals.astype(np.int16), return_counts=True)
        counts[uniq] += cnt.astype(np.int16)
    return counts

def fill_by_iterative_neighbors(major, target_mask, reference_lc=None, reference_weight=0, max_radius_px=EGIS_MAX_FILL_RADIUS_PX):
    filled = major.copy().astype(np.int16)
    pending = target_mask.astype(bool).copy()
    fill_mask = np.zeros_like(pending, dtype=bool)
    reference_vote_mask = np.zeros_like(pending, dtype=bool)
    radius_used = np.zeros(filled.shape, dtype=np.int16)
    for radius in range(1, int(max_radius_px) + 1):
        ys, xs = np.where(pending)
        if len(ys) == 0:
            break
        updates = []
        used_reference = []
        for y, x in zip(ys, xs):
            counts = neighbor_counts(filled, int(y), int(x))
            ref_used = False
            if reference_lc is not None:
                ref_cls = int(reference_lc[int(y), int(x)])
                if ref_cls > 0:
                    counts[ref_cls] += int(reference_weight)
                    ref_used = True
            winner = choose_weighted_majority(counts)
            if winner > 0:
                updates.append((int(y), int(x), int(winner)))
                used_reference.append(ref_used)
        if not updates:
            break
        for (y, x, winner), ref_used in zip(updates, used_reference):
            filled[y, x] = winner
            pending[y, x] = False
            fill_mask[y, x] = True
            reference_vote_mask[y, x] = bool(ref_used)
            radius_used[y, x] = radius
    return filled, fill_mask, reference_vote_mask, pending, radius_used

def fill_by_expanding_window(major, target_mask, max_radius_px=EGIS_MAX_FILL_RADIUS_PX):
    filled = major.copy().astype(np.int16)
    pending = target_mask.astype(bool).copy()
    fill_mask = np.zeros_like(pending, dtype=bool)
    radius_used = np.zeros(filled.shape, dtype=np.int16)
    for radius in range(1, int(max_radius_px) + 1):
        ys, xs = np.where(pending)
        if len(ys) == 0:
            break
        updates = []
        for y, x in zip(ys, xs):
            y0, y1 = max(0, int(y) - radius), min(filled.shape[0], int(y) + radius + 1)
            x0, x1 = max(0, int(x) - radius), min(filled.shape[1], int(x) + radius + 1)
            vals = filled[y0:y1, x0:x1].reshape(-1)
            vals = vals[vals > 0]
            if vals.size == 0:
                continue
            counts = np.zeros(max(HYBRID_CLASSES) + 1, dtype=np.int16)
            uniq, cnt = np.unique(vals.astype(np.int16), return_counts=True)
            counts[uniq] += cnt.astype(np.int16)
            winner = choose_weighted_majority(counts)
            if winner > 0:
                updates.append((int(y), int(x), int(winner)))
        if not updates:
            continue
        for y, x, winner in updates:
            filled[y, x] = winner
            pending[y, x] = False
            fill_mask[y, x] = True
            radius_used[y, x] = radius
    return filled, fill_mask, pending, radius_used

def auto_fill_egis_unknown(major, target_mask, analysis_year):
    year = int(analysis_year)
    stats = {
        "egis_fill_method": "majority",
        "egis_reference_fill_year": -1,
        "egis_reference_fill_weight": 0,
        "egis_reference_vote_pixels": 0,
        "egis_majority_fill_pixels": 0,
        "egis_manual_fill_pixels": 0,
        "egis_fill_max_radius_px": 0,
        "egis_fill_max_radius_m": 0.0,
        "egis_fill_radius_cap_px": int(EGIS_MAX_FILL_RADIUS_PX),
        "egis_fill_radius_cap_m": float(EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M),
        "egis_unknown_leftover_pixels": 0,
        "egis_reference_raw_npz": "",
    }
    if not bool(target_mask.any()):
        stats["egis_fill_method"] = "none"
        return major.copy().astype(np.int16), stats
    reference_lc = None
    if year <= EGIS_REFERENCE_FILL_MAX_YEAR:
        ref_policy = egis_policy_for_year(EGIS_REFERENCE_YEAR)
        ref_policy, ref_rgba, ref_major, ref_raw_code, ref_source_nodata, ref_unrecognized, ref_color_diag = egis_direct_for_policy(ref_policy, grid)
        ref_unknown = (ref_major == 0) | ref_source_nodata.astype(bool) | ref_unrecognized.astype(bool)
        if bool(ref_unknown.any()):
            ref_major, ref_majority_mask, ref_leftover, ref_radius = fill_by_expanding_window(ref_major, ref_unknown, EGIS_MAX_FILL_RADIUS_PX)
            if bool(ref_leftover.any()):
                print(f"2022 reference still has {int(ref_leftover.sum())} unknown pixels after radius cap; those pixels are ignored as reference votes.")
                ref_major[ref_leftover] = 0
        ref_npz = DIRECT_RAW_DIR / f"egis_direct_raw_{RUN_LABEL}_reference_{ref_policy['source_key']}.npz"
        np.savez_compressed(
            ref_npz,
            raw_rgba=ref_rgba,
            egis_hybrid_lc_ref_filled_60m=ref_major.astype(np.int16),
            egis_raw_code_60m=ref_raw_code.astype(np.int16),
            egis_source_nodata_mask=ref_source_nodata.astype(np.uint8),
            egis_unrecognized_rgb_mask=ref_unrecognized.astype(np.uint8),
            reference_year=np.array(int(EGIS_REFERENCE_YEAR)),
            reference_fill_radius_cap_px=np.array(int(EGIS_MAX_FILL_RADIUS_PX)),
            **{f"reference_{k}": v for k, v in ref_color_diag.items()},
            **{k: np.array(v) for k, v in ref_policy.items()},
        )
        reference_lc = ref_major.astype(np.int16)
        stats.update({
            "egis_fill_method": "attempt3_8nbr_plus_2022x4",
            "egis_reference_fill_year": int(EGIS_REFERENCE_YEAR),
            "egis_reference_fill_weight": int(EGIS_REFERENCE_FILL_WEIGHT),
            "egis_reference_raw_npz": str(ref_npz),
        })
    filled, neighbor_mask, ref_vote_mask, pending, radius_used = fill_by_iterative_neighbors(
        major,
        target_mask,
        reference_lc=reference_lc,
        reference_weight=EGIS_REFERENCE_FILL_WEIGHT if reference_lc is not None else 0,
        max_radius_px=EGIS_MAX_FILL_RADIUS_PX,
    )
    if bool(pending.any()):
        # Final bounded window majority for islands that had no 8-neighbor propagation path.
        filled, window_mask, pending, window_radius = fill_by_expanding_window(filled, pending, EGIS_MAX_FILL_RADIUS_PX)
        radius_used = np.maximum(radius_used, window_radius)
        neighbor_mask = neighbor_mask | window_mask
        stats["egis_fill_method"] += "_plus_bounded_window_majority"
    stats["egis_reference_vote_pixels"] = int(ref_vote_mask.sum())
    stats["egis_majority_fill_pixels"] = int(neighbor_mask.sum())
    stats["egis_unknown_leftover_pixels"] = int(pending.sum())
    stats["egis_fill_max_radius_px"] = int(radius_used.max()) if radius_used.size else 0
    stats["egis_fill_max_radius_m"] = float(stats["egis_fill_max_radius_px"] * EGIS_PIXEL_SIZE_M)
    if bool(pending.any()):
        raise RuntimeError(
            f"EGIS auto fill left {int(pending.sum())} unknown pixels after {EGIS_MAX_FILL_RADIUS_PX}px "
            f"({EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M:.0f}m) cap. Rerun with manual sea/urban or increase EGIS_MAX_FILL_RADIUS_PX."
        )
    return filled.astype(np.int16), stats

def apply_egis_fill(major, target_mask, choice, fill_class):
    if choice == "none":
        stats = {
            "egis_fill_method": "none",
            "egis_reference_fill_year": -1,
            "egis_reference_fill_weight": 0,
            "egis_reference_vote_pixels": 0,
            "egis_majority_fill_pixels": 0,
            "egis_manual_fill_pixels": 0,
            "egis_fill_max_radius_px": 0,
            "egis_fill_max_radius_m": 0.0,
            "egis_fill_radius_cap_px": int(EGIS_MAX_FILL_RADIUS_PX),
            "egis_fill_radius_cap_m": float(EGIS_MAX_FILL_RADIUS_PX * EGIS_PIXEL_SIZE_M),
            "egis_unknown_leftover_pixels": 0,
            "egis_reference_raw_npz": "",
        }
        return major.copy().astype(np.int16), stats
    if choice == "auto":
        return auto_fill_egis_unknown(major, target_mask, analysis_year)
    raise RuntimeError("EGIS fill은 attempt3 자동 보간만 허용합니다. 바다/도심 강제 채움은 사용하지 않습니다.")


fill_choice, fill_class = read_fill_choice()
filled_lc, egis_fill_stats = apply_egis_fill(egis_major, unknown_mask, fill_choice, fill_class)
record_fill_decision(fill_choice, fill_class, egis_fill_stats)
print("EGIS fill stats:", egis_fill_stats)
egis_arr, egis_names = egis_onehot_from_lc(filled_lc)
condition = np.concatenate([terrain, albedo, aws, egis_arr], axis=0).astype(np.float32)
condition_channels = TERRAIN_BANDS + ALBEDO_BANDS + AWS_BANDS + egis_names
if condition_channels != CONDITION_CHANNELS:
    raise RuntimeError("condition channel order mismatch")
meta = sample_meta(landsat_info["selected_landsat_date"], row["sun_azimuth"], row["sun_elevation"], ROI_CENTER_LAT, ROI_CENTER_LON)

out = OUT_DIR / f"{RUN_LABEL}_{landsat_info['selected_landsat_product_id']}_condition.npz"
if out.exists() and not OVERWRITE:
    print("exists:", out)
else:
    np.savez_compressed(
        out,
        condition=condition,
        condition_channels=np.array(condition_channels),
        lst_k_raw_direct=lst_k_raw_direct.astype(np.float32),
        lst_clear_mask_direct=lst_clear_mask_direct.astype(np.uint8),
        lst_cloud_fill_mask_direct=lst_cloud_fill_mask_direct.astype(np.uint8),
        meta=meta,
        meta_channels=np.array(META_CHANNELS),
        run_label=np.array(RUN_LABEL),
        roi_label=np.array(ROI_LABEL),
        data_generation_source=np.array("direct_ee_kma_egis_wms"),
        requested_date=np.array(landsat_info["requested_date"]),
        date=np.array(landsat_info["selected_landsat_date"]),
        selected_landsat_date=np.array(landsat_info["selected_landsat_date"]),
        landsat_exact_available=np.array(bool(landsat_info["landsat_exact_available"])),
        landsat_exact_count=np.array(int(landsat_info["landsat_exact_count"])),
        selected_landsat_product_id=np.array(landsat_info["selected_landsat_product_id"]),
        selected_landsat_scene_id=np.array(landsat_info["selected_landsat_scene_id"]),
        selected_landsat_collection=np.array(landsat_info["selected_landsat_collection"]),
        latitude=np.array(float(ROI_CENTER_LAT), dtype=np.float32),
        longitude=np.array(float(ROI_CENTER_LON), dtype=np.float32),
        year=np.array(int(analysis_year)),
        sun_azimuth=np.array(float(row["sun_azimuth"]), dtype=np.float32),
        sun_elevation=np.array(float(row["sun_elevation"]), dtype=np.float32),
        aws_source=np.array("KMA_API_direct"),
        aws_tm=np.array(str(aws_tm)),
        aws_station_count=np.array(int(len(aws_station_frame))),
        terrain_source=np.array("EE_USGS_SRTMGL1_003_direct"),
        albedo_source=np.array("EE_Landsat_attempt3_clear_mask_focal_mean_local_expanding_mean"),
        **{k: np.array(v) for k, v in albedo_fill_stats.items()},
        egis_source=np.array("EGIS_WMS_direct"),
        egis_policy_year=np.array(int(egis_policy["policy_year"])),
        egis_source_key=np.array(str(egis_policy["source_key"])),
        egis_source_level=np.array(str(egis_policy["source_level"])),
        egis_raw_direct_npz=np.array(str(raw_egis_npz)),
        egis_unknown_pixels=np.array(int(unknown_mask.sum())),
        egis_unknown_ratio=np.array(float(unknown_mask.mean()), dtype=np.float32),
        egis_unknown_fill_choice=np.array("auto_attempt3" if fill_choice == "auto" else str(fill_choice)),
        egis_unknown_fill_class=np.array(-1),
        egis_source_nodata_water_fill_mask=np.zeros_like(egis_source_nodata, dtype=np.uint8),
        egis_unknown_review_png=np.array(str(preview_png)),
        meta_norm_info=np.array(json.dumps({"source": META_NORM_SOURCE, "lat_mean": LAT_META_MEAN, "lat_std": LAT_META_STD, "lon_mean": LON_META_MEAN, "lon_std": LON_META_STD}, ensure_ascii=False)),
        **{k: np.array(v) for k, v in egis_unrecognized_fill_stats.items()},
        **{k: np.array(v) for k, v in egis_fill_stats.items()},
    )
    manifest_row = {
        "output": str(out),
        "run_label": RUN_LABEL,
        "roi_label": ROI_LABEL,
        "data_generation_source": "direct_ee_kma_egis_wms",
        "requested_date": landsat_info["requested_date"],
        "selected_landsat_date": landsat_info["selected_landsat_date"],
        "landsat_exact_available": bool(landsat_info["landsat_exact_available"]),
        "landsat_exact_count": int(landsat_info["landsat_exact_count"]),
        "selected_landsat_product_id": landsat_info["selected_landsat_product_id"],
        "selected_landsat_scene_id": landsat_info["selected_landsat_scene_id"],
        "lst_clear_pixels": int(lst_clear_mask_direct.sum()),
        "lst_clear_fraction": float(lst_clear_mask_direct.mean()),
        "roi_center_lat": float(ROI_CENTER_LAT),
        "roi_center_lon": float(ROI_CENTER_LON),
        "aws_source": "KMA_API_direct",
        "aws_tm": str(aws_tm),
        "aws_station_count": int(len(aws_station_frame)),
        "egis_source": "EGIS_WMS_direct",
        "egis_source_key": egis_policy["source_key"],
        "egis_unrecognized_pixels": int(egis_unrecognized_mask.sum()),
        "egis_unknown_pixels": int(unknown_mask.sum()),
        "egis_unknown_fill_choice": "auto_attempt3" if fill_choice == "auto" else fill_choice,
        "egis_source_nodata_water_fill_pixels": 0,
        "unknown_review_png": str(preview_png),
    }
    manifest_row.update(albedo_fill_stats)
    manifest_row.update(egis_unrecognized_fill_stats)
    manifest_row.update(egis_fill_stats)
    manifest = pd.DataFrame([manifest_row])
    manifest.to_csv(OUT_DIR / "condition_manifest.csv", index=False, encoding="utf-8-sig")
    print("saved condition:", out)
    print("manifest:", OUT_DIR / "condition_manifest.csv")
out


# Condition 생성 직후 시각화까지 바로 저장/표시한다.
def scalar_value(value):
    arr = np.asarray(value)
    return arr.item() if arr.shape == () else arr


def colormap_rgba(arr, cmap_name="turbo", vmin=None, vmax=None, percentiles=(2, 98)):
    import matplotlib.cm as mpl_cm
    values = np.asarray(arr, dtype=np.float32)
    finite = np.isfinite(values)
    rgba = np.zeros((*values.shape, 4), dtype=np.uint8)
    rgba[..., 3] = 255
    rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)
    if finite.any():
        if vmin is None or vmax is None:
            lo, hi = np.nanpercentile(values[finite], percentiles)
            if vmin is None:
                vmin = float(lo)
            if vmax is None:
                vmax = float(hi)
        denom = max(float(vmax - vmin), 1e-6)
        normed = np.clip((values - float(vmin)) / denom, 0, 1)
        cmap = mpl_cm.get_cmap(cmap_name)
        rgba = (cmap(normed) * 255).astype(np.uint8)
        rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)
    return rgba


def condition_egis_class_map(condition, channels):
    egis_idx = [i for i, name in enumerate(channels) if str(name).startswith('egis_cat_')]
    if not egis_idx:
        return None
    egis_stack = condition[egis_idx]
    class_values = []
    for idx in egis_idx:
        name = str(channels[idx])
        class_values.append(int(name.split('_')[2]))
    winner = np.argmax(egis_stack, axis=0)
    lc = np.zeros(egis_stack.shape[1:], dtype=np.int16)
    for k, cls in enumerate(class_values):
        lc[winner == k] = int(cls)
    return lc


def final_condition_visualization(npz_path):
    with np.load(npz_path, allow_pickle=True) as z:
        cond = z['condition'].astype(np.float32)
        channels = [str(x) for x in z['condition_channels']]
        requested = scalar_value(z.get('requested_date', ''))
        selected = scalar_value(z.get('selected_landsat_date', ''))
        roi = scalar_value(z.get('roi_label', ROI_LABEL))
    tiles = []
    for name in ['elevation', 'slope', 'hillshade_landsat', 'albedo', 'avg_temp', 'rain_1h', 'humidity']:
        if name in channels:
            tiles.append((name, colormap_rgba(cond[channels.index(name)], cmap_name='turbo')))
    lc = condition_egis_class_map(cond, channels)
    if lc is not None:
        tiles.append(('egis filled classes', hybrid_to_rgba(lc)))
    if not tiles:
        raise RuntimeError('No visualizable condition channels found')

    labeled = []
    for title, rgba in tiles:
        img = Image.fromarray(rgba).convert('RGBA').resize((256, 256), Image.Resampling.NEAREST)
        labeled.append(add_label(img, title))
    cols = 4
    rows = int(math.ceil(len(labeled) / cols))
    title_h = 30
    sheet = Image.new('RGBA', (cols * 256, rows * 280 + title_h), (255, 255, 255, 255))
    draw = ImageDraw.Draw(sheet)
    draw.text((6, 8), f'ROI={roi} | requested={requested} | selected_landsat={selected}', fill=(0, 0, 0, 255))
    for i, img in enumerate(labeled):
        sheet.paste(img, ((i % cols) * 256, title_h + (i // cols) * 280))
    quicklook = VIS_DIR / f'{RUN_LABEL}_condition_quicklook.png'
    sheet.save(quicklook)
    return quicklook

condition_quicklook_png = final_condition_visualization(out)
print('condition quicklook:', condition_quicklook_png)
try:
    from IPython.display import Image as DisplayImage, display
    display(DisplayImage(filename=str(condition_quicklook_png)))
except Exception:
    pass


## 2. CVAE LST 복원 생성


In [ ]:
from pathlib import Path
import json, math
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# 01에서 만든 같은 RUN_ROOT_DIR/condition 결과를 그대로 이어서 복원한다.
CKPT_NAME = "best_balanced.pt"
USE_PRIOR_MEAN = True
N_RANDOM_SAMPLES = 0        # 0이면 prior mean만 저장. 불확실성 샘플이 필요하면 8 등으로 설정
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONDITION_DIR = RUN_ROOT_DIR / "outputs" / "condition"
STRUCTURED_DIR = CONDITION_DIR
RECONSTRUCTION_DIR = RUN_ROOT_DIR / "outputs" / "reconstruction"
OUT_DIR = RECONSTRUCTION_DIR
VIS_DIR = RUN_ROOT_DIR / "outputs" / "visualization"
CKPT_PATH = RECONST_DIR / "savedpt" / CKPT_NAME
for p in [OUT_DIR, VIS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
if not CKPT_PATH.exists():
    raise FileNotFoundError(f"checkpoint missing: {CKPT_PATH}")
print("run_label:", RUN_LABEL)
print("run root:", RUN_ROOT_DIR)
print("device:", DEVICE)
print("structured:", STRUCTURED_DIR)
print("reconstruction output:", OUT_DIR)
print("visualization output:", VIS_DIR)
print("checkpoint:", CKPT_PATH)


def scalar(value):
    arr = np.asarray(value)
    return arr.item() if arr.shape == () else arr

def scalar_str(value):
    return str(scalar(value))

def minmax(values, channel_stats, names):
    out = values.astype(np.float32, copy=True)
    for i, name in enumerate(names):
        lo = channel_stats[name]["min"]
        hi = channel_stats[name]["max"]
        denom = hi - lo
        out[i] = 0.0 if abs(denom) <= 1e-12 else (out[i] - lo) / denom
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def denormalize(values, channel_stats, names):
    out = values.astype(np.float32, copy=True)
    for i, name in enumerate(names):
        lo = channel_stats[name]["min"]
        hi = channel_stats[name]["max"]
        out[i] = out[i] * (hi - lo) + lo
    return out.astype(np.float32)


def channel_indices(names, predicates):
    out = []
    for i, name in enumerate(names):
        low = name.lower()
        if any(pred(low) for pred in predicates):
            out.append(i)
    return out

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_ch, out_ch, 4, 2, 1), nn.GroupNorm(min(8, out_ch), out_ch), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1), nn.GroupNorm(min(8, out_ch), out_ch), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class FiLM(nn.Module):
    def __init__(self, context_dim, ch):
        super().__init__()
        self.to_gamma_beta = nn.Linear(context_dim, ch * 2)
    def forward(self, x, context):
        gb = self.to_gamma_beta(context)
        gamma, beta = gb.chunk(2, dim=1)
        return x * (1 + gamma[:, :, None, None]) + beta[:, :, None, None]

class ConditionEncoder(nn.Module):
    def __init__(self, in_ch, width):
        super().__init__()
        self.s0 = ConvBlock(in_ch, width)
        self.d1 = Down(width, width)
        self.d2 = Down(width, width * 2)
        self.d3 = Down(width * 2, width * 4)
        self.d4 = Down(width * 4, width * 8)
        self.d5 = Down(width * 8, width * 8)
    def forward(self, x):
        s0 = self.s0(x); s1 = self.d1(s0); s2 = self.d2(s1); s3 = self.d3(s2); s4 = self.d4(s3); b = self.d5(s4)
        return b, [s0, s1, s2, s3, s4]

class Decoder(nn.Module):
    def __init__(self, bottleneck_ch, out_ch, width, context_dim, latent_ch=0, gated_skips=True):
        super().__init__()
        self.gated_skips = gated_skips
        in_ch = bottleneck_ch + latent_ch
        self.u4 = Up(in_ch, width * 8); self.f4 = ConvBlock(width * 16, width * 8); self.film4 = FiLM(context_dim, width * 8)
        self.u3 = Up(width * 8, width * 4); self.f3 = ConvBlock(width * 8, width * 4); self.film3 = FiLM(context_dim, width * 4)
        self.u2 = Up(width * 4, width * 2); self.f2 = ConvBlock(width * 4, width * 2); self.film2 = FiLM(context_dim, width * 2)
        self.u1 = Up(width * 2, width); self.f1 = ConvBlock(width * 2, width); self.film1 = FiLM(context_dim, width)
        self.u0 = Up(width, width); self.f0 = ConvBlock(width * 2, width); self.film0 = FiLM(context_dim, width)
        self.out = nn.Conv2d(width, out_ch, 3, padding=1)
        if gated_skips:
            self.skip_gates = nn.Parameter(torch.full((5,), -0.5))
    def gate(self, skip, idx):
        if not self.gated_skips:
            return skip
        return skip * torch.sigmoid(self.skip_gates[idx])
    def forward(self, bottleneck, skips, context, latent_map=None):
        x = torch.cat([bottleneck, latent_map], dim=1) if latent_map is not None else bottleneck
        s0, s1, s2, s3, s4 = skips
        x = self.u4(x); x = self.f4(torch.cat([x, self.gate(s4, 4)], dim=1)); x = self.film4(x, context)
        x = self.u3(x); x = self.f3(torch.cat([x, self.gate(s3, 3)], dim=1)); x = self.film3(x, context)
        x = self.u2(x); x = self.f2(torch.cat([x, self.gate(s2, 2)], dim=1)); x = self.film2(x, context)
        x = self.u1(x); x = self.f1(torch.cat([x, self.gate(s1, 1)], dim=1)); x = self.film1(x, context)
        x = self.u0(x); x = self.f0(torch.cat([x, self.gate(s0, 0)], dim=1)); x = self.film0(x, context)
        return self.out(x)

class PosteriorEncoder(nn.Module):
    def __init__(self, in_ch, width):
        super().__init__()
        self.net = nn.Sequential(ConvBlock(in_ch, width), Down(width, width), Down(width, width * 2), Down(width * 2, width * 4), Down(width * 4, width * 8), Down(width * 8, width * 8))
    def forward(self, condition, target):
        return self.net(torch.cat([condition, target], dim=1))

class GaussianHead(nn.Module):
    def __init__(self, in_dim, hidden_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.SiLU(inplace=True), nn.Linear(hidden_dim, hidden_dim), nn.SiLU(inplace=True))
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h).clamp(-12, 8)

class MultiLatentConditionalPriorResidualCVAE(nn.Module):
    def __init__(self, condition_ch, target_ch, meta_dim, latent_blocks, width=32, grid_n=256, condition_channels=None, meta_channels=None):
        super().__init__()
        condition_channels = condition_channels or []
        meta_channels = meta_channels or []
        self.weather_idx = channel_indices(condition_channels, [lambda s: s in {'avg_temp', 'humidity', 'wind_u', 'wind_v', 'rain_1h'}])
        self.land_idx = channel_indices(condition_channels, [lambda s: s == 'albedo', lambda s: s.startswith('egis_cat_')])
        self.static_idx = channel_indices(condition_channels, [lambda s: s in {'elevation', 'slope', 'aspect_sin', 'aspect_cos', 'hillshade_landsat'}])
        meta_index = {name: i for i, name in enumerate(meta_channels)}
        self.season_meta_idx = [meta_index[x] for x in ['day_sin', 'day_cos', 'sun_azimuth_norm', 'sun_elevation_norm'] if x in meta_index]
        self.global_meta_idx = [meta_index[x] for x in ['lat_norm', 'lon_norm'] if x in meta_index]
        reduced = grid_n // 32
        flat = width * 8 * reduced * reduced
        self.latent_blocks = dict(latent_blocks)
        self.latent_dim = sum(self.latent_blocks.values())
        self.width = width
        self.reduced = reduced
        self.condition_encoder = ConditionEncoder(condition_ch, width)
        self.posterior_encoder = PosteriorEncoder(condition_ch + target_ch, width)
        self.context_head = nn.Sequential(nn.Linear(width * 8 + meta_dim, width * 8), nn.SiLU(inplace=True), nn.Linear(width * 8, width * 8), nn.SiLU(inplace=True))
        pooled_dim = width * 8
        self.block_input_dims = {
            'global': pooled_dim + len(self.static_idx) + len(self.global_meta_idx),
            'season': pooled_dim + len(self.season_meta_idx),
            'weather': pooled_dim + len(self.weather_idx) + len(self.season_meta_idx),
            'land': pooled_dim + len(self.land_idx) + len(self.global_meta_idx),
            'residual': pooled_dim + meta_dim,
        }
        post_shared_dim = width * 16
        self.post_shared = nn.Sequential(nn.Linear(flat + meta_dim, post_shared_dim), nn.SiLU(inplace=True), nn.Linear(post_shared_dim, post_shared_dim), nn.SiLU(inplace=True))
        self.prior_heads = nn.ModuleDict({name: GaussianHead(self.block_input_dims[name], width * 8, dim) for name, dim in self.latent_blocks.items()})
        self.posterior_heads = nn.ModuleDict({name: GaussianHead(post_shared_dim + self.block_input_dims[name], width * 8, dim) for name, dim in self.latent_blocks.items()})
        self.z_to_map = nn.Sequential(nn.Linear(self.latent_dim + width * 8, flat), nn.SiLU(inplace=True))
        self.base_decoder = Decoder(width * 8, target_ch, width, context_dim=width * 8, latent_ch=0, gated_skips=True)
        self.residual_decoder = Decoder(width * 8, target_ch, width, context_dim=width * 8, latent_ch=width * 8, gated_skips=True)
    def channel_summary(self, condition, idxs):
        if not idxs:
            return condition.new_zeros((condition.shape[0], 0))
        return condition[:, idxs].mean(dim=(2, 3))
    def meta_select(self, meta, idxs):
        if not idxs:
            return meta.new_zeros((meta.shape[0], 0))
        return meta[:, idxs]
    def block_inputs(self, condition, bottleneck, meta):
        pooled = bottleneck.mean(dim=(2, 3))
        return {
            'global': torch.cat([pooled, self.channel_summary(condition, self.static_idx), self.meta_select(meta, self.global_meta_idx)], dim=1),
            'season': torch.cat([pooled, self.meta_select(meta, self.season_meta_idx)], dim=1),
            'weather': torch.cat([pooled, self.channel_summary(condition, self.weather_idx), self.meta_select(meta, self.season_meta_idx)], dim=1),
            'land': torch.cat([pooled, self.channel_summary(condition, self.land_idx), self.meta_select(meta, self.global_meta_idx)], dim=1),
            'residual': torch.cat([pooled, meta], dim=1),
        }
    def context(self, bottleneck, meta):
        pooled = bottleneck.mean(dim=(2, 3))
        return self.context_head(torch.cat([pooled, meta], dim=1))
    def prior(self, condition, bottleneck, meta):
        inputs = self.block_inputs(condition, bottleneck, meta)
        return {name: self.prior_heads[name](inputs[name]) for name in self.latent_blocks}
    def posterior(self, condition, target, bottleneck, meta):
        post_feat = self.posterior_encoder(condition, target).flatten(1)
        shared = self.post_shared(torch.cat([post_feat, meta], dim=1))
        inputs = self.block_inputs(condition, bottleneck, meta)
        return {name: self.posterior_heads[name](torch.cat([shared, inputs[name]], dim=1)) for name in self.latent_blocks}
    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
    def sample_z(self, stats, use_mean=False):
        parts = []
        for name in self.latent_blocks:
            mu, logvar = stats[name]
            parts.append(mu if use_mean else self.reparameterize(mu, logvar))
        return torch.cat(parts, dim=1)
    def decode(self, condition, meta, z=None, use_prior_mean=False):
        b, skips = self.condition_encoder(condition)
        ctx = self.context(b, meta)
        prior_stats = self.prior(condition, b, meta)
        if z is None:
            z = self.sample_z(prior_stats, use_mean=use_prior_mean)
        zmap = self.z_to_map(torch.cat([z, ctx], dim=1)).view(condition.shape[0], self.width * 8, self.reduced, self.reduced)
        base = self.base_decoder(b, skips, ctx)
        residual = self.residual_decoder(b, skips, ctx, zmap)
        return base + residual, base, residual, prior_stats


def load_model_from_ckpt(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    condition_channels = [str(x) for x in ckpt["condition_channels"]]
    target_channels = [str(x) for x in ckpt["target_channels"]]
    meta_channels = [str(x) for x in ckpt["meta_channels"]]
    latent_blocks = ckpt.get("latent_blocks", {"global": 24, "season": 16, "weather": 16, "land": 24, "residual": 16})
    width = int(ckpt.get("width", 32))
    grid_n = int(ckpt.get("grid_n", 256))
    model = MultiLatentConditionalPriorResidualCVAE(
        len(condition_channels), len(target_channels), len(meta_channels),
        latent_blocks, width=width, grid_n=grid_n,
        condition_channels=condition_channels, meta_channels=meta_channels,
    ).to(DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print("loaded:", ckpt_path.name, "epoch=", ckpt.get("epoch"), "metrics=", ckpt.get("metrics", {}))
    return ckpt, model

ckpt, model = load_model_from_ckpt(CKPT_PATH)


def read_condition_for_ckpt(path, ckpt):
    stats = ckpt["normalization_stats"]
    condition_channels = [str(x) for x in ckpt["condition_channels"]]
    meta_channels = [str(x) for x in ckpt["meta_channels"]]
    with np.load(path, allow_pickle=True) as z:
        z_condition_channels = [str(x) for x in z["condition_channels"]]
        z_meta_channels = [str(x) for x in z["meta_channels"]]
        if z_condition_channels != condition_channels:
            raise RuntimeError(f"condition channel mismatch: {path.name}")
        if z_meta_channels != meta_channels:
            raise RuntimeError(f"meta channel mismatch: {path.name}")
        condition = minmax(z["condition"].astype(np.float32), stats["condition"], condition_channels)
        meta = z["meta"].astype(np.float32)
        attrs = {k: z[k] for k in z.files if k not in {"condition", "condition_channels", "meta", "meta_channels"}}
    return torch.from_numpy(condition[None]).float(), torch.from_numpy(meta[None]).float(), attrs

def colormap_rgba(arr, cmap_name="turbo", vmin=None, vmax=None, percentiles=(2, 98)):
    import matplotlib.cm as mpl_cm
    values = np.asarray(arr, dtype=np.float32)
    finite = np.isfinite(values)
    rgba = np.zeros((*values.shape, 4), dtype=np.uint8)
    rgba[..., 3] = 255
    rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)
    if finite.any():
        if vmin is None or vmax is None:
            lo, hi = np.nanpercentile(values[finite], percentiles)
            if vmin is None:
                vmin = float(lo)
            if vmax is None:
                vmax = float(hi)
        denom = max(float(vmax - vmin), 1e-6)
        normed = np.clip((values - float(vmin)) / denom, 0, 1)
        cmap = mpl_cm.get_cmap(cmap_name)
        rgba = (cmap(normed) * 255).astype(np.uint8)
        rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)
    return rgba


def heatmap_rgba(arr):
    return colormap_rgba(arr, cmap_name="turbo")

def add_label(img, label, height=24):
    from PIL import Image, ImageDraw
    base = Image.new("RGBA", (img.width, img.height + height), (255, 255, 255, 255))
    base.paste(img, (0, height))
    draw = ImageDraw.Draw(base)
    draw.text((4, 5), str(label)[:80], fill=(0, 0, 0, 255))
    return base

def save_prediction_png(pred, target_channels, out_png, title):
    from PIL import Image, ImageDraw
    tiles = []
    for name, arr in zip(target_channels, pred):
        img = Image.fromarray(heatmap_rgba(arr)).resize((256, 256), Image.Resampling.NEAREST)
        tiles.append(add_label(img, name))
    title_h = 28
    sheet = Image.new("RGBA", (256 * len(tiles), 256 + 24 + title_h), (255, 255, 255, 255))
    draw = ImageDraw.Draw(sheet)
    draw.text((4, 6), str(title)[:140], fill=(0, 0, 0, 255))
    for i, tile in enumerate(tiles):
        sheet.paste(tile, (i * 256, title_h))
    sheet.save(out_png)

def clear_preserved_outputs(pred, attrs, min_clear_pixels=512):
    raw = np.asarray(attrs.get("lst_k_raw_direct"), dtype=np.float32) if "lst_k_raw_direct" in attrs else None
    clear = np.asarray(attrs.get("lst_clear_mask_direct"), dtype=bool) if "lst_clear_mask_direct" in attrs else None
    if raw is None or clear is None:
        return pred.astype(np.float32), pred.astype(np.float32), pred.astype(np.float32), np.float32(0.0), np.array(False)
    valid_clear = clear & np.isfinite(raw)
    bias = np.float32(0.0)
    used_bias = np.array(False)
    if pred.shape[0] > 0 and int(valid_clear.sum()) >= int(min_clear_pixels):
        diff = pred[0][valid_clear] - raw[valid_clear]
        diff = diff[np.isfinite(diff)]
        if diff.size >= int(min_clear_pixels):
            bias = np.float32(np.nanmedian(diff))
            used_bias = np.array(True)
    corrected = (pred - bias).astype(np.float32)
    clear_preserved = pred.astype(np.float32).copy()
    corrected_clear_preserved = corrected.copy()
    for i in range(pred.shape[0]):
        clear_preserved[i][valid_clear] = raw[valid_clear]
        corrected_clear_preserved[i][valid_clear] = raw[valid_clear]
    return clear_preserved, corrected, corrected_clear_preserved, bias, used_bias

def run_one(path):
    out_npz = OUT_DIR / f"{path.stem}_lst_reconst.npz"
    out_png = OUT_DIR / f"{path.stem}_lst_reconst.png"
    if out_npz.exists() and not OVERWRITE:
        return {"condition": str(path), "output": str(out_npz), "png": str(out_png), "status": "SKIP_EXISTING"}
    cond, meta, attrs = read_condition_for_ckpt(path, ckpt)
    cond = cond.to(DEVICE)
    meta = meta.to(DEVICE)
    with torch.no_grad():
        pred_n, base_n, residual_n, prior_stats = model.decode(cond, meta, use_prior_mean=USE_PRIOR_MEAN)
        pred = denormalize(pred_n.cpu().numpy()[0], ckpt["normalization_stats"]["target"], [str(x) for x in ckpt["target_channels"]])
        base = denormalize(base_n.cpu().numpy()[0], ckpt["normalization_stats"]["target"], [str(x) for x in ckpt["target_channels"]])
        residual = residual_n.cpu().numpy()[0].astype(np.float32)
        random_preds = []
        for _ in range(int(N_RANDOM_SAMPLES)):
            sample_n, _, _, _ = model.decode(cond, meta, use_prior_mean=False)
            random_preds.append(denormalize(sample_n.cpu().numpy()[0], ckpt["normalization_stats"]["target"], [str(x) for x in ckpt["target_channels"]]))
    target_channels = np.array([str(x) for x in ckpt["target_channels"]])
    payload = dict(attrs)
    clear_preserved, bias_corrected, bias_corrected_clear_preserved, clear_bias_k, clear_bias_used = clear_preserved_outputs(pred, payload)
    save_prediction_png(bias_corrected_clear_preserved, target_channels, out_png, f"final clear-preserved | roi={scalar_str(attrs.get('roi_label', ''))} | requested={scalar_str(attrs.get('requested_date', ''))} | selected_landsat={scalar_str(attrs.get('selected_landsat_date', ''))}")
    payload.update({
        "prediction": pred.astype(np.float32),
        "prediction_clear_preserved": clear_preserved.astype(np.float32),
        "prediction_bias_corrected": bias_corrected.astype(np.float32),
        "prediction_bias_corrected_clear_preserved": bias_corrected_clear_preserved.astype(np.float32),
        "clear_pixel_model_bias_k": np.array(float(clear_bias_k), dtype=np.float32),
        "clear_pixel_bias_correction_used": np.array(bool(clear_bias_used)),
        "base": base.astype(np.float32),
        "residual_normalized": residual.astype(np.float32),
        "target_channels": target_channels,
        "checkpoint": np.array(str(CKPT_PATH)),
        "use_prior_mean": np.array(bool(USE_PRIOR_MEAN)),
    })
    if random_preds:
        payload["random_predictions"] = np.stack(random_preds).astype(np.float32)
    np.savez_compressed(out_npz, **payload)
    return {
        "condition": str(path),
        "output": str(out_npz),
        "png": str(out_png),
        "status": "OK",
        "run_label": scalar_str(attrs.get("run_label", RUN_LABEL)),
        "data_generation_source": scalar_str(attrs.get("data_generation_source", "")),
        "roi_label": scalar_str(attrs.get("roi_label", "")),
        "requested_date": scalar_str(attrs.get("requested_date", "")),
        "landsat_exact_available": scalar_str(attrs.get("landsat_exact_available", "")),
        "landsat_exact_count": scalar_str(attrs.get("landsat_exact_count", "")),
        "selected_landsat_date": scalar_str(attrs.get("selected_landsat_date", "")),
        "template_date": scalar_str(attrs.get("template_date", "")),
        "template_date_delta_days": scalar_str(attrs.get("template_date_delta_days", "")),
        "clear_pixel_model_bias_k": float(clear_bias_k),
        "clear_pixel_bias_correction_used": bool(clear_bias_used),
    }

condition_files = sorted(STRUCTURED_DIR.glob("*_condition.npz"))
if not condition_files:
    raise FileNotFoundError(f"condition npz missing: {STRUCTURED_DIR}. Run 01 first.")
rows = [run_one(path) for path in condition_files]
manifest = pd.DataFrame(rows)
manifest.to_csv(OUT_DIR / "reconstruction_manifest.csv", index=False, encoding="utf-8-sig")
print(manifest.to_string(index=False))
print("manifest:", OUT_DIR / "reconstruction_manifest.csv")


# Reconstruction 생성 직후 quicklook까지 바로 저장한다.
def scalar_value(value):
    arr = np.asarray(value)
    return arr.item() if arr.shape == () else arr


def npz_scalar(z, key, default=""):
    return scalar_value(z[key]) if key in z.files else default


def colormap_rgba(arr, cmap_name="turbo", vmin=None, vmax=None, percentiles=(2, 98)):
    import matplotlib.cm as mpl_cm
    values = np.asarray(arr, dtype=np.float32)
    finite = np.isfinite(values)
    rgba = np.zeros((*values.shape, 4), dtype=np.uint8)
    rgba[..., 3] = 255
    rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)
    if finite.any():
        if vmin is None or vmax is None:
            lo, hi = np.nanpercentile(values[finite], percentiles)
            if vmin is None:
                vmin = float(lo)
            if vmax is None:
                vmax = float(hi)
        denom = max(float(vmax - vmin), 1e-6)
        normed = np.clip((values - float(vmin)) / denom, 0, 1)
        cmap = mpl_cm.get_cmap(cmap_name)
        rgba = (cmap(normed) * 255).astype(np.uint8)
        rgba[~finite] = np.array([0, 0, 0, 255], dtype=np.uint8)
    return rgba


def lst_rgba(arr, vmin=None, vmax=None):
    return colormap_rgba(arr, cmap_name="turbo", vmin=vmin, vmax=vmax)


def diff_rgba(arr, limit=6.0):
    return colormap_rgba(arr, cmap_name="coolwarm", vmin=-float(limit), vmax=float(limit))


def shared_limits(arrays, percentiles=(2, 98), min_span=2.0):
    vals = []
    for arr in arrays:
        values = np.asarray(arr, dtype=np.float32)
        finite = np.isfinite(values)
        if bool(finite.any()):
            vals.append(values[finite].reshape(-1))
    if not vals:
        return None, None
    vals = np.concatenate(vals)
    lo, hi = np.nanpercentile(vals, percentiles)
    if float(hi - lo) < float(min_span):
        mid = 0.5 * float(lo + hi)
        lo = mid - float(min_span) / 2
        hi = mid + float(min_span) / 2
    return float(lo), float(hi)


def diff_limit(arr, default=6.0, min_limit=3.0, max_limit=15.0):
    values = np.asarray(arr, dtype=np.float32)
    finite = np.isfinite(values)
    if not bool(finite.any()):
        return float(default)
    lim = float(np.nanpercentile(np.abs(values[finite]), 98))
    return float(np.clip(lim, min_limit, max_limit))


def error_label(name, diff):
    finite = np.isfinite(diff)
    if not bool(finite.any()):
        return f"{name} | no common clear pixels"
    vals = diff[finite]
    rmse = float(np.sqrt(np.nanmean(vals ** 2)))
    mae = float(np.nanmean(np.abs(vals)))
    bias = float(np.nanmean(vals))
    return f"{name} | RMSE={rmse:.2f}K MAE={mae:.2f}K bias={bias:.2f}K n={int(finite.sum())}"


def add_label_local(img, label, height=24):
    from PIL import Image, ImageDraw
    base = Image.new("RGBA", (img.width, img.height + height), (255, 255, 255, 255))
    base.paste(img, (0, height))
    draw = ImageDraw.Draw(base)
    draw.text((4, 5), str(label)[:90], fill=(0, 0, 0, 255))
    return base


def lst_label(name, arr):
    finite = np.isfinite(arr)
    if finite.any():
        return f"{name} | min={np.nanmin(arr):.2f}K max={np.nanmax(arr):.2f}K valid={int(finite.sum())}"
    return f"{name} | no valid pixels"


def build_reconstruction_quicklook(npz_path):
    from PIL import Image, ImageDraw
    npz_path = Path(npz_path)
    with np.load(npz_path, allow_pickle=True) as z:
        pred = z["prediction"].astype(np.float32)
        final_pred = z["prediction_bias_corrected_clear_preserved"].astype(np.float32) if "prediction_bias_corrected_clear_preserved" in z.files else z["prediction_clear_preserved"].astype(np.float32) if "prediction_clear_preserved" in z.files else pred
        target_channels = [str(x) for x in z["target_channels"]]
        requested = npz_scalar(z, "requested_date", "")
        selected = npz_scalar(z, "selected_landsat_date", "")
        roi = npz_scalar(z, "roi_label", ROI_LABEL)
        product = npz_scalar(z, "selected_landsat_product_id", "")
        raw_lst = z["lst_k_raw_direct"].astype(np.float32) if "lst_k_raw_direct" in z.files else None
        raw_clear = z["lst_clear_mask_direct"].astype(bool) if "lst_clear_mask_direct" in z.files else None
        bias = float(npz_scalar(z, "clear_pixel_model_bias_k", 0.0))
        bias_used = bool(npz_scalar(z, "clear_pixel_bias_correction_used", False))

    raw_display = None
    if raw_lst is not None:
        raw_display = raw_lst.copy()
        if raw_clear is not None:
            raw_display[~raw_clear] = np.nan
    vmin, vmax = shared_limits(([raw_display] if raw_display is not None else []) + [pred[0], final_pred[0]])

    tiles = []
    for i, name in enumerate(target_channels):
        img = Image.fromarray(lst_rgba(final_pred[i], vmin=vmin, vmax=vmax)).resize((256, 256), Image.Resampling.NEAREST)
        tiles.append(add_label_local(img, lst_label(f"final clear-preserved {name}", final_pred[i])))
        img = Image.fromarray(lst_rgba(pred[i], vmin=vmin, vmax=vmax)).resize((256, 256), Image.Resampling.NEAREST)
        tiles.append(add_label_local(img, lst_label(f"model prior {name}", pred[i])))
        if raw_display is not None and i == 0:
            diff = np.where(np.isfinite(raw_display), pred[i] - raw_display, np.nan).astype(np.float32)
            lim = diff_limit(diff)
            img = Image.fromarray(diff_rgba(diff, limit=lim)).resize((256, 256), Image.Resampling.NEAREST)
            tiles.append(add_label_local(img, error_label(f"model prior - observed clear", diff)))
    cols = min(3, max(1, len(tiles)))
    rows = int(np.ceil(len(tiles) / cols))
    title_h = 42
    sheet = Image.new("RGBA", (cols * 256, rows * 280 + title_h), (255, 255, 255, 255))
    draw = ImageDraw.Draw(sheet)
    scale_text = "shared turbo scale" if vmin is None else f"shared turbo scale {vmin:.1f}-{vmax:.1f}K"
    draw.text((6, 8), f"ROI={roi} | requested={requested} | selected_landsat={selected} | {product}", fill=(0, 0, 0, 255))
    draw.text((6, 24), scale_text + f" | final keeps observed clear pixels | prior bias={bias:.2f}K used={bias_used}", fill=(0, 0, 0, 255))
    for i, img in enumerate(tiles):
        sheet.paste(img, ((i % cols) * 256, title_h + (i // cols) * 280))
    quicklook = VIS_DIR / f"{npz_path.stem}_quicklook.png"
    sheet.save(quicklook)
    return quicklook


def final_reconstruction_visualization(manifest_path):
    df = pd.read_csv(manifest_path)
    ok = df[df["status"].astype(str).isin(["OK", "SKIP_EXISTING"])].copy()
    if ok.empty:
        raise RuntimeError(f"No reconstruction outputs in {manifest_path}")
    quicklooks = []
    for _, item in ok.iterrows():
        quicklooks.append(build_reconstruction_quicklook(Path(item["output"])))
    return quicklooks

reconstruction_manifest_path = OUT_DIR / "reconstruction_manifest.csv"
reconstruction_quicklook_pngs = final_reconstruction_visualization(reconstruction_manifest_path)
try:
    from IPython.display import Image as DisplayImage, display
except Exception:
    DisplayImage = None
    display = None
for quicklook in reconstruction_quicklook_pngs:
    print("reconstruction quicklook:", quicklook)
    if display is not None:
        display(DisplayImage(filename=str(quicklook)))
